# Chapter 5 Block Matrices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asj252/linear-algebra-with-python/blob/main/en/EN_Chap5_Block_Matrix.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/asj252/linear-algebra-with-python/HEAD?labpath=en%2FEN_Chap5_Block_Matrix.ipynb)

In 1858, in *A Memoir on the Theory of Matrices*, Arthur Cayley systematically treated the matrix as a complete algebraic entity—not a collection of coefficients taken from a system of equations, but a "single object" that can be added, multiplied, and inverted. This shift in viewpoint leads naturally to a question: **if this object is overwhelmingly large, can we cut it into pieces and perform algebraic operations directly on those pieces, just as we would with ordinary numbers?**

Over the following century, this question spread from the halls of abstract algebra to the computational work of science and engineering, and in the end it profoundly reshaped the underlying architecture of modern computing.

The Schur complement shows the power of block operations. Let the block matrix with compatible sizes be

$$\mathbf{M} = \begin{pmatrix} \mathbf{A} & \mathbf{B} \\ \mathbf{C} & \mathbf{D} \end{pmatrix}$$

If $\mathbf{A}$ is an invertible square matrix, block Gaussian elimination gives $\det\mathbf{M}=\det\mathbf{A}\det(\mathbf{D}-\mathbf{C}\mathbf{A}^{-1}\mathbf{B})$. Here $\mathbf{S}=\mathbf{D}-\mathbf{C}\mathbf{A}^{-1}\mathbf{B}$ is called the Schur complement. **When the sizes are compatible and the blocks that must be inverted are invertible, submatrices can serve as units of computation; but matrix multiplication is in general not commutative, so the order of the factors cannot be changed at will.** This structure has important applications in multivariate Gaussian distributions, control theory, and convex optimization.

As problems in science and engineering grew in scale, "blocking" also became an important way to organize computation: divide a large matrix into manageable sub-blocks, schedule the movement of data, distribute the work, and then combine the results according to compatible rules of operation. Today's distributed computing and CPU cache blocking still exploit this idea. The concrete benefits depend on the data layout, the hardware, and the algorithm; partitioning into blocks does not by itself guarantee fewer arithmetic operations.

In 1969, Volker Strassen showed that partitioning into blocks can also change the number of operations. The standard algorithm needs $n^3$ scalar multiplications to compute the product of two $n\times n$ matrices, but this is not a proven lower bound. He constructed a formula for $2\times2$ block multiplication that needs only **7** block multiplications instead of the 8 required by direct expansion.

Precisely because the objects being operated on inside the blocks are still matrices, the technique can be applied recursively down to scalars or to suitably small blocks, lowering the computational complexity of matrix multiplication directly from $O(n^3)$ to $O(n^{\log_2 7}) \approx O(n^{2.807})$. Strassen's paper bore a strikingly provocative title: *Gaussian Elimination is not Optimal*. This was not merely a matter of saving a few multiplications; it changed how people understood the operation count of the standard algorithm and set off a "gold rush" around the matrix multiplication exponent $\omega$ that is still being fought out at the research frontier today.

The story of block matrices teaches us that **structure is power**. When the dimension of a problem is hopelessly high, seeing the block geometry inside it is like holding the key that cuts the problem down to size. Partitioning into blocks is not a passive compromise but a profound way of thinking that makes complex large-scale systems modular and parallel—from the stiffness assembly of finite element analysis and the partial trace of density matrices in quantum mechanics to the tensor blocking of the self-attention mechanism in today's deep-learning Transformer architectures, this key keeps turning in the deepest layers of the computational world.

### Chapter Structure and Learning Objectives

Cayley's idea of "the matrix as an object" is the prerequisite for block thinking, the Schur complement is an important application of it, and Strassen's algorithm is its most counterintuitive demonstration. The task of this chapter is to place this way of thinking on a rigorous algebraic foundation.

§5.1 first establishes the language: the definition of block matrices, together with addition, scalar multiplication, and transposition. The rules for these basic operations are entirely analogous to those for ordinary matrices, but they call for care with one extra condition, the "compatibility of block sizes."

§5.2 is the core of the chapter: block multiplication. Two block matrices are multiplied by a rule that has exactly the same form as ordinary matrix multiplication—simply replace "entries" by "submatrix blocks"—provided that the dimensions of adjacent blocks match. Strassen's algorithm was born within the framework of this section; its formula with 7 multiplications looks like magic, yet careful checking shows it to be exactly right.

§5.3 discusses inversion of block matrices. For a $2 \times 2$ block structure, the Schur complement provides an elegant inversion formula and at the same time reveals the precise relationship between "the large matrix is invertible" and "certain sub-blocks are invertible."

§5.4 turns to advanced topics: the tensor product (Kronecker product) and the partial trace. The tensor product "weaves" two matrices into a larger matrix and is the mathematical foundation both of composite systems in quantum mechanics and of the multi-head attention mechanism in deep learning; the partial trace is a reduction operation that eliminates the degrees of freedom of one subsystem, is in general not invertible, and is used to "extract" information about a subsystem from a composite system.

Once you have read this chapter, your fear of "large matrices" should be somewhat smaller—because you will know that a large matrix is nothing more than a few small matrices put together according to a pattern.

---


## 5.1 Definition and Basic Operations of Block Matrices

A block matrix is a large matrix divided into several smaller matrix blocks, which can be regarded as the "sub-entries" of the matrix. Partitioning in this way turns operations on a large matrix into operations on the individual small matrices, thereby simplifying the computation.

::: {prf:definition} Block Matrix
:label: def-block-matrix
A block matrix is a matrix divided into a number of smaller matrix blocks; each block is a part of the matrix and can independently take part in operations such as addition, scalar multiplication, and transposition. For example, a matrix can be divided into four blocks: upper left, upper right, lower left, and lower right.
:::

Below we introduce three basic operations on block matrices.

### 5.1.1 Addition of Block Matrices

If two block matrices have the same block structure, their sum is defined by adding corresponding blocks. Let

$$
\mathbf{A} = \begin{bmatrix} \mathbf{A}_{00} & \mathbf{A}_{01} \\ \mathbf{A}_{10} & \mathbf{A}_{11} \end{bmatrix}, \quad
\mathbf{B} = \begin{bmatrix} \mathbf{B}_{00} & \mathbf{B}_{01} \\ \mathbf{B}_{10} & \mathbf{B}_{11} \end{bmatrix},
$$
Then

$$
\mathbf{A} + \mathbf{B} = \begin{bmatrix} \mathbf{A}_{00}+\mathbf{B}_{00} & \mathbf{A}_{01}+\mathbf{B}_{01} \\[1mm] \mathbf{A}_{10}+\mathbf{B}_{10} & \mathbf{A}_{11}+\mathbf{B}_{11} \end{bmatrix}.
$$

::: {prf:example} Block Matrix Addition
:label: ex-block-addition
Let

$$
\textcolor{#57068C}{\mathbf{A}_{00} = \begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix}},\quad
\textcolor{#006385}{\mathbf{A}_{01} = \begin{bmatrix} 5 & 6 \\ 7 & 8 \end{bmatrix}},\quad
\textcolor{#007A87}{\mathbf{A}_{10} = \begin{bmatrix} 9 & 10 \\ 11 & 12 \end{bmatrix}},\quad
\textcolor{#FF8C00}{\mathbf{A}_{11} = \begin{bmatrix} 13 & 14 \\ 15 & 16 \end{bmatrix}}
$$

$$
\textcolor{#57068C}{\mathbf{B}_{00} = \begin{bmatrix} a & b \\ c & d \end{bmatrix}},\quad
\textcolor{#006385}{\mathbf{B}_{01} = \begin{bmatrix} e & f \\ g & h \end{bmatrix}},\quad
\textcolor{#007A87}{\mathbf{B}_{10} = \begin{bmatrix} m & n \\ o & p \end{bmatrix}},\quad
\textcolor{#FF8C00}{\mathbf{B}_{11} = \begin{bmatrix} q & r \\ s & t \end{bmatrix}}
$$

Regarding them as $2\times 2$ block matrices, we have

$$
\mathbf{A}+\mathbf{B} = \begin{bmatrix}
\textcolor{#57068C}{1+a}  & \textcolor{#57068C}{2+b}  & \textcolor{#006385}{5+e}  & \textcolor{#006385}{6+f}  \\
\textcolor{#57068C}{3+c}  & \textcolor{#57068C}{4+d}  & \textcolor{#006385}{7+g}  & \textcolor{#006385}{8+h}  \\
\textcolor{#007A87}{9+m}  & \textcolor{#007A87}{10+n} & \textcolor{#FF8C00}{13+q} & \textcolor{#FF8C00}{14+r} \\
\textcolor{#007A87}{11+o} & \textcolor{#007A87}{12+p} & \textcolor{#FF8C00}{15+s} & \textcolor{#FF8C00}{16+t}
\end{bmatrix}.
$$

:::

### 5.1.2 Scalar Multiplication of Block Matrices

To perform **scalar multiplication** on a block matrix, simply multiply each block by the scalar. That is, if $c$ is a scalar,

$$
c\mathbf{A} = \begin{bmatrix} c\mathbf{A}_{00} & c\mathbf{A}_{01} \\[1mm] c\mathbf{A}_{10} & c\mathbf{A}_{11} \end{bmatrix}.
$$

### 5.1.3 Transpose of Block Matrices

Taking the **transpose** of a block matrix requires both transposing each block and swapping the positions of the blocks. Writing $\mathbf{A}^\top$ for the transpose of $\mathbf{A}$, we have

$$
\mathbf{A}^\top = \begin{bmatrix} (\mathbf{A}_{00})^\top & (\mathbf{A}_{10})^\top \\[1mm] (\mathbf{A}_{01})^\top & (\mathbf{A}_{11})^\top \end{bmatrix}.
$$

::: {prf:example} Block Matrix Transpose
:label: ex-block-transpose
Let

$$
\mathbf{A} = \begin{bmatrix} \begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix} & \begin{bmatrix} 5 \\ 6 \end{bmatrix} \\[2mm] \begin{bmatrix} 7 & 8 \end{bmatrix} & 9 \end{bmatrix},
$$
Then

$$
\mathbf{A}^\top = \begin{bmatrix} \begin{bmatrix} 1 & 3 \\ 2 & 4 \end{bmatrix} & \begin{bmatrix} 7 \\ 8 \end{bmatrix} \\[2mm] \begin{bmatrix} 5 & 6 \end{bmatrix} & 9 \end{bmatrix}.
$$
:::

---

### 5.1.4 ◆4D Tensor Representation and Indexing of Block Matrices

Besides being viewed as a two-dimensional structure made up of small matrix blocks, a block matrix can also be regarded as a **4D tensor**; this viewpoint is important in modern machine learning and quantum mechanics. With the 4D tensor representation we can handle batched block matrix operations more naturally and take advantage of the tensor operations offered by modern computing frameworks.

::::{prf:definition} 4D Tensor Representation of a Block Matrix
:label: def-block-matrix-4d-tensor


Suppose the block matrix $\mathbf{A}$ has an $m \times n$ block structure and each block  $\mathbf{A}_{ij}$ has size $p \times q$. Then it can be represented as a 4D tensor $\mathcal{A} \in \mathbb{R}^{m \times n \times p \times q}$ with entries

$$\mathcal{A}[i, j, k, l] := [\mathbf{A}_{ij}]_{kl}$$

**Meaning of the axes.**
- **Axis 0 ($i$)**: block-row index $(0 \leq i < m)$
- **Axis 1 ($j$)**: block-column index $(0 \leq j < n)$  
- **Axis 2 ($k$)**: row index of the entry within the block $(0 \leq k < p)$
- **Axis 3 ($l$)**: column index of the entry within the block $(0 \leq l < q)$
::::


For a $4 \times 4$ matrix partitioned into a $2 \times 2$ block structure, we have the following correspondence:

**2D matrix index**: $\mathbf{A}[r, c]$, where $0 \leq r, c < 4$

**4D tensor index**: $\mathcal{A}[i, j, k, l]$, where
- $i = \lfloor r/2 \rfloor$ (block-row index)
- $j = \lfloor c/2 \rfloor$ (block-column index) 
- $k = r \operatorname{mod} 2$ (within-block row index)
- $l = c \operatorname{mod} 2$ (within-block column index)

::::{prf:example}
:label: ex-4d-tensor-indexing
**Index correspondence.**

Consider the $4 \times 4$ block matrix

$$
\mathbf{A} = \begin{bmatrix}
\textcolor{#57068C}{1}  & \textcolor{#57068C}{2}  & \textcolor{#006385}{5}  & \textcolor{#006385}{6}  \\
\textcolor{#57068C}{3}  & \textcolor{#57068C}{4}  & \textcolor{#006385}{7}  & \textcolor{#006385}{8}  \\
\textcolor{#007A87}{9}  & \textcolor{#007A87}{10} & \textcolor{#FF8C00}{13} & \textcolor{#FF8C00}{14} \\
\textcolor{#007A87}{11} & \textcolor{#007A87}{12} & \textcolor{#FF8C00}{15} & \textcolor{#FF8C00}{16}
\end{bmatrix}
$$


The corresponding 4D tensor representation:

| 2D index | Entry | 4D index | Description |
|--------|--------|--------|------|
| `A[0,0]` | 1 | $\mathcal{A}[0,0,0,0]$ | upper-left entry of the upper-left block |
| `A[0,1]` | 2 | $\mathcal{A}[0,0,0,1]$ | upper-right entry of the upper-left block |
| `A[0,2]` | 5 | $\mathcal{A}[0,1,0,0]$ | upper-left entry of the upper-right block |
| `A[1,3]` | 8 | $\mathcal{A}[0,1,1,1]$ | lower-right entry of the upper-right block |
| `A[2,0]` | 9 | $\mathcal{A}[1,0,0,0]$ | upper-left entry of the lower-left block |
| `A[3,3]` | 16 | $\mathcal{A}[1,1,1,1]$ | lower-right entry of the lower-right block |
::::



**Summary.**  
Representing a block matrix as a 4D tensor gives us a powerful tool, especially in modern computing environments that call for large numbers of block-level operations. Understanding this representation and its indexing rules helps us design and implement block matrix algorithms more effectively.

### 5.1.5 Python: Block Matrix Conversions with NumPy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 60)
print("5.1 Building block matrices - using np.block, np.stack, np.hstack")
print("=" * 60)

# ========== 1. Defining the basic blocks ==========
print("\n1. Define the basic matrix blocks")
print("-" * 30)

# Define four 2×2 submatrix blocks
A00 = np.array([[1, 2],
                [3, 4]])
A01 = np.array([[5, 6],
                [7, 8]])
A10 = np.array([[9, 10],
                [11, 12]])
A11 = np.array([[13, 14],
                [15, 16]])

print(f"A00 (upper-left block):\n{A00}")
print(f"\nA01 (upper-right block):\n{A01}")
print(f"\nA10 (lower-left block):\n{A10}")
print(f"\nA11 (lower-right block):\n{A11}")

# ========== 2. Assembling block matrices with np.block ==========
print("\n\n2. Assemble a block matrix with np.block")
print("-" * 40)

# np.block is the most intuitive way to build a block matrix
block_matrix = np.block([[A00, A01],
                         [A10, A11]])

print("Using np.block([[A00, A01], [A10, A11]]):")
print(block_matrix)
print(f"Shape: {block_matrix.shape}")

# A more complex block structure
print("\nComplex block structure - zero and identity matrices:")
I2 = np.eye(2)  # 2×2 identity matrix
O2 = np.zeros((2, 2))  # 2×2 zero matrix

complex_block = np.block([[A00, O2, A01],
                          [I2, A10, O2],
                          [A11, I2, A00]])

print(f"Shape: {complex_block.shape}")
print(complex_block)

# Combining blocks of different sizes
print("\nCombining blocks of different sizes:")
B1 = np.ones((3, 2))    # 3×2 all-ones matrix
B2 = np.eye(3)          # 3×3 identity matrix
B3 = np.zeros((2, 2))   # 2×2 zero matrix
B4 = np.full((2, 3), 5) # 2×3 matrix filled with 5

mixed_block = np.block([[B2, B1],
                        [B4, B3]])
print(f"Shape of the mixed-size block matrix: {mixed_block.shape}")
print(mixed_block)

# ========== 3. Horizontal stacking with np.hstack ==========
print("\n\n3. Horizontal stacking with np.hstack")
print("-" * 35)

# First assemble the top and bottom rows separately
top_row = np.hstack([A00, A01])
bottom_row = np.hstack([A10, A11])

print("Top row np.hstack([A00, A01]):")
print(top_row)
print(f"Shape: {top_row.shape}")

print("\nBottom row np.hstack([A10, A11]):")
print(bottom_row)
print(f"Shape: {bottom_row.shape}")

# Then stack vertically
hstack_result = np.vstack([top_row, bottom_row])
print("\nFinal result np.vstack([top_row, bottom_row]):")
print(hstack_result)
print(f"Shape: {hstack_result.shape}")

# Check agreement with the np.block result
print(f"\nAgrees with the np.block result: {np.array_equal(block_matrix, hstack_result)}")

# Building in several steps
print("\nBuilding a large block matrix in several steps:")
# Create more blocks
C11 = np.full((2, 2), 17)
C12 = np.full((2, 2), 18)
C21 = np.full((2, 2), 19)
C22 = np.full((2, 2), 20)

# Step 1: combine horizontally
row1 = np.hstack([A00, A01, C11, C12])
row2 = np.hstack([A10, A11, C21, C22])
print(f"Shape of block row 0: {row1.shape}")
print(f"Shape of block row 1: {row2.shape}")

# Step 2: combine vertically
large_matrix = np.vstack([row1, row2])
print(f"Shape of the large matrix: {large_matrix.shape}")

# ========== 4. Using np.stack to help build a 2D block matrix ==========
print("\n\n4. Using np.stack to help build a 2D block matrix")
print("-" * 45)

# Method 1: organize the rows with np.stack, then reshape to 2D
print("Method 1: stack into rows first, then reshape")
top_blocks = [A00, A01]
bottom_blocks = [A10, A11] 

# Stack along the row direction
top_row_stack = np.stack(top_blocks, axis=1)  # 2×2×2 -> reshaped to 2×4
bottom_row_stack = np.stack(bottom_blocks, axis=1)  # 2×2×2 -> reshaped to 2×4

print(f"Shape of the stacked top row: {top_row_stack.shape}")
print(f"Shape of the stacked bottom row: {bottom_row_stack.shape}")

# Reshape to 2D
top_row_2d = top_row_stack.reshape(2, 4)
bottom_row_2d = bottom_row_stack.reshape(2, 4)

print("Top row reshaped to 2D:")
print(top_row_2d)
print("Bottom row reshaped to 2D:")
print(bottom_row_2d)

# Final vertical combination
stack_result = np.vstack([top_row_2d, bottom_row_2d])
print("\nFinal 2D block matrix:")
print(stack_result)
print(f"Shape: {stack_result.shape}")

# Method 2: organize everything with np.stack, then reshape
print("\nMethod 2: stack everything, then reshape")
# Arrange in the logical order of the block matrix
blocks_2x2 = np.array([[A00, A01],
                       [A10, A11]])  # shape: (2, 2, 2, 2)

print(f"Shape of the 4D block structure: {blocks_2x2.shape}")

# Reshape into a 2D block matrix
stack_method2 = np.concatenate([
    np.concatenate([blocks_2x2[0, 0], blocks_2x2[0, 1]], axis=1),
    np.concatenate([blocks_2x2[1, 0], blocks_2x2[1, 1]], axis=1)
], axis=0)

print("Reshaped 2D matrix:")
print(stack_method2)

# Method 3: handle different kinds of blocks with np.stack
print("\nMethod 3: handling blocks of mixed kinds")
# Create different kinds of 2×2 blocks
zero_block = np.zeros((2, 2))
ones_block = np.ones((2, 2))
eye_block = np.eye(2)
random_block = np.random.randint(1, 5, (2, 2))

print("Zero block:")
print(zero_block)
print("All-ones block:")
print(ones_block)
print("Identity block:")
print(eye_block)
print("Random block:")
print(random_block)

# Organize these different blocks with stack
special_blocks = [zero_block, ones_block, eye_block, random_block]

# Create a 2×2 block structure
mixed_top = np.hstack([special_blocks[0], special_blocks[1]])
mixed_bottom = np.hstack([special_blocks[2], special_blocks[3]])
mixed_result = np.vstack([mixed_top, mixed_bottom])

print("\nBlock matrix with mixed kinds of blocks:")
print(mixed_result)
print(f"Shape: {mixed_result.shape}")

# Check that all methods give consistent results
print(f"\nMethod 1 equals the original block_matrix: {np.array_equal(stack_result, block_matrix)}")
print(f"Method 2 equals the original block_matrix: {np.array_equal(stack_method2, block_matrix)}")


# ========== 5. Converting a 2D tensor into a 4D tensor ==========
print("\n\n5. Converting a 2D tensor into a 4D tensor")
print("-" * 35)

print("Original 2D block matrix:")
print(f"Shape: {block_matrix.shape}")
print(block_matrix)

# Method 1: convert the block matrix into a 4D tensor with reshape
# Assume the original matrix is 4×4 and consists of 2×2 blocks; convert it to (2,2,2,2)
tensor_4d_method1 = block_matrix.reshape(2, 2, 2, 2).transpose(0, 2, 1, 3)
for i in range(2):
    for j in range(2):
        assert np.array_equal(tensor_4d_method1[i, j], block_matrix[2*i:2*i+2, 2*j:2*j+2])
print(f"\nMethod 1 - reshape: {block_matrix.shape} -> {tensor_4d_method1.shape}")
print("Block [0,0,:,:] of the 4D tensor:")
print(tensor_4d_method1[0, 0])
print("Block [0,1,:,:] of the 4D tensor:")
print(tensor_4d_method1[0, 1])

# Method 2: regroup manually into a 4D tensor, keeping the meaning of the block structure
print(f"\nMethod 2 - regrouping the block structure manually:")
# Extract each 2×2 block
block_00 = block_matrix[0:2, 0:2]  # A00
block_01 = block_matrix[0:2, 2:4]  # A01  
block_10 = block_matrix[2:4, 0:2]  # A10
block_11 = block_matrix[2:4, 2:4]  # A11

# Combine into a 4D tensor (block row, block column, within-block row, within-block column)
tensor_4d_method2 = np.array([[block_00, block_01],
                              [block_10, block_11]])

print(f"Shape of the 4D tensor: {tensor_4d_method2.shape}")
print("Block [0,1] (corresponds to A01):")
print(tensor_4d_method2[0, 1])
print("Block [1,1] (corresponds to A11):")
print(tensor_4d_method2[1, 1])

# Method 3: rebuild the block structure by slicing
print(f"\nMethod 3 - building the 4D tensor by slicing:")
# A more flexible slicing approach, for equal-sized blocks that divide the matrix evenly
def create_4d_tensor_from_blocks(matrix, block_rows, block_cols):
    """
    Convert a 2D block matrix into a 4D tensor
    matrix: the input 2D matrix
    block_rows, block_cols: the number of rows and the number of columns of each block
    """
    matrix = np.asarray(matrix)
    if matrix.ndim != 2:
        raise ValueError("matrix must be a two-dimensional matrix")
    if any(isinstance(x, (bool, np.bool_)) or not isinstance(x, (int, np.integer)) or x <= 0 for x in (block_rows, block_cols)):
        raise ValueError("the block sizes must be positive integers")
    rows, cols = matrix.shape
    if rows % block_rows or cols % block_cols:
        raise ValueError("both dimensions of the matrix must be divisible by the block sizes")
    n_block_rows = rows // block_rows
    n_block_cols = cols // block_cols
    
    # Create the 4D tensor
    tensor_4d = np.zeros((n_block_rows, n_block_cols, block_rows, block_cols), dtype=matrix.dtype)
    
    for i in range(n_block_rows):
        for j in range(n_block_cols):
            start_row = i * block_rows
            end_row = (i + 1) * block_rows
            start_col = j * block_cols
            end_col = (j + 1) * block_cols
            tensor_4d[i, j] = matrix[start_row:end_row, start_col:end_col]
    
    return tensor_4d

tensor_4d_method3 = create_4d_tensor_from_blocks(block_matrix, 2, 2)
print(f"By slicing: {block_matrix.shape} -> {tensor_4d_method3.shape}")
print("Block [0,0] (corresponds to A00):")
print(tensor_4d_method3[0, 0])
print("Block [1,1] (corresponds to A11):")
print(tensor_4d_method3[1, 1])

# Method 4: partition into blocks with np.split
print(f"\nMethod 4 - partitioning with np.split:")
# First split by rows
row_splits = np.split(block_matrix, 2, axis=0)  # split into 2 rows
print(f"Splitting by rows gives {len(row_splits)} submatrices, each of shape: {row_splits[0].shape}")

# Then split each row by columns
blocks_list = []
for row_block in row_splits:
    col_splits = np.split(row_block, 2, axis=1)  # split each row into 2 columns
    blocks_list.append(col_splits)

# Convert to a 4D tensor
tensor_4d_method4 = np.array(blocks_list)
print(f"Using split: {block_matrix.shape} -> {tensor_4d_method4.shape}")
print("Block [0,0] (corresponds to A00):")
print(tensor_4d_method4[0, 0])
print("Block [1,1] (corresponds to A11):")
print(tensor_4d_method4[1, 1])

# ========== 6. Converting a 4D tensor back into a 2D tensor ==========
print("\n\n6. Converting a 4D tensor back into a 2D tensor")
print("-" * 30)

print("Now we convert the 4D tensor back into a 2D block matrix")

# Use the 4D tensor generated above (taking method2 as an example)
print(f"Shape of the original 4D tensor: {tensor_4d_method2.shape}")
print("Contents of the 4D tensor:")
print("Block [0,0]:", tensor_4d_method2[0,0])
print("Block [0,1]:", tensor_4d_method2[0,1])
print("Block [1,0]:", tensor_4d_method2[1,0])
print("Block [1,1]:", tensor_4d_method2[1,1])

# Method 1: restore the axis order with transpose, then reshape
print(f"\nMethod 1 - using reshape:")
# Reshape from (2,2,2,2) to (4,4)
reconstructed_1 = tensor_4d_method1.transpose(0, 2, 1, 3).reshape(4, 4)
print(f"Shape conversion: {tensor_4d_method1.shape} -> {reconstructed_1.shape}")
print("Reconstructed 2D matrix:")
print(reconstructed_1)

# Method 2: using swapaxes + reshape
print(f"\nMethod 2 - using swapaxes + reshape:")
# Swap axes first, then reshape
swapped = tensor_4d_method2.swapaxes(1, 2)  # (2,2,2,2) -> (2,2,2,2), but rearranged
print(f"Shape after swapping axes: {swapped.shape}")
reconstructed_2 = swapped.reshape(4, 4)
print("Reconstructed 2D matrix:")
print(reconstructed_2)

# Method 3: regroup manually (the clearest method)
print(f"\nMethod 3 - regrouping the blocks manually:")
# Extract each block
block_00 = tensor_4d_method3[0, 0]  # upper left
block_01 = tensor_4d_method3[0, 1]  # upper right
block_10 = tensor_4d_method3[1, 0]  # lower left
block_11 = tensor_4d_method3[1, 1]  # lower right

# Regroup with hstack and vstack
top_row = np.hstack([block_00, block_01])
bottom_row = np.hstack([block_10, block_11])
reconstructed_3 = np.vstack([top_row, bottom_row])

print("Reconstructed 2D matrix:")
print(reconstructed_3)

# Method 4: using np.block (the most concise)
print(f"\nMethod 4 - using np.block:")
reconstructed_4 = np.block([[tensor_4d_method4[0,0], tensor_4d_method4[0,1]],
                            [tensor_4d_method4[1,0], tensor_4d_method4[1,1]]])
print("Reconstructed 2D matrix:")
print(reconstructed_4)

# Method 5: using concatenate
print(f"\nMethod 5 - using concatenate:")
# First join the blocks of each row along axis 3 (within-block columns)
row_0 = np.concatenate([tensor_4d_method2[0,0], tensor_4d_method2[0,1]], axis=1)
row_1 = np.concatenate([tensor_4d_method2[1,0], tensor_4d_method2[1,1]], axis=1)
# Then join all the rows along axis 2 (within-block rows)
reconstructed_5 = np.concatenate([row_0, row_1], axis=0)
print("Reconstructed 2D matrix:")
print(reconstructed_5)

# General function: 4D tensor to 2D matrix
def tensor_4d_to_2d(tensor_4d):
    """
    Convert a 4D tensor (block row, block column, within-block row, within-block column) into a 2D matrix
    """
    n_block_rows, n_block_cols, block_rows, block_cols = tensor_4d.shape
    
    # Approach: use reshape together with an axis permutation
    # First permute to (block row, within-block row, block column, within-block column)
    reshaped = tensor_4d.transpose(0, 2, 1, 3)
    # Then reshape to 2D
    result = reshaped.reshape(n_block_rows * block_rows, n_block_cols * block_cols)
    
    return result

print(f"\nTesting the general function:")
reconstructed_general = tensor_4d_to_2d(tensor_4d_method2)
print("2D matrix reconstructed by the general function:")
print(reconstructed_general)

# Check the correctness of all methods
print(f"\nVerification results:")
print(f"Original matrix:")
print(block_matrix)

all_methods = [reconstructed_1, reconstructed_3, reconstructed_4, reconstructed_5, reconstructed_general]
method_names = ["reshape", "manual regrouping", "np.block", "concatenate", "general function"]

for i, (method_result, name) in enumerate(zip(all_methods, method_names)):
    is_equal = np.array_equal(block_matrix, method_result)
    print(f"{name}: equals the original matrix = {is_equal}")

# Special case: handling 4D tensors of other sizes
print(f"\nHandling 4D tensors of other sizes:")
# Create an example 4D tensor with 3×3 blocks
large_blocks = []
for i in range(3):
    row_blocks = []
    for j in range(3):
        # each block is 2×2
        block = np.full((2, 2), i*3 + j + 1)
        row_blocks.append(block)
    large_blocks.append(row_blocks)

large_4d = np.array(large_blocks)
print(f"Shape of the large 4D tensor: {large_4d.shape}")

# Convert to a 6×6 2D matrix
large_2d = tensor_4d_to_2d(large_4d)
print(f"Shape of the converted 2D matrix: {large_2d.shape}")
print("Converted 2D matrix:")
print(large_2d)

print("\n" + "=" * 60)
print("Summary of methods for building and converting block matrices:")
print("1. np.block: the most intuitive way to assemble blocks")
print("2. np.hstack/vstack: horizontal/vertical stacking, suited to step-by-step construction")
print("3. np.stack: stacking along a new axis, creating a structure for batch processing")
print("4. 2D→4D conversion: reshape, manual regrouping, slicing")
print("=" * 60)

:::{admonition} §5.1 Summary
:class: tip
This section answers the most basic questions about block matrices: what does it mean to partition a matrix into blocks, and how are algebraic operations on blocks carried out correctly? A block matrix is a representation in which a large matrix is systematically cut into a number of submatrix blocks, and the rules for addition, scalar multiplication, and transposition are entirely analogous to those for ordinary matrices—addition requires the two block matrices to have the same block structure, while transposition both swaps the positions of the blocks and transposes each block individually; the "compatibility of block sizes" is an implicit condition running through all three operations.

This section also introduced the 4D tensor viewpoint, which gives the block indices and the within-block entry indices of a block structure a single unified address $\mathcal{A}[i,j,k,l]$. This is not merely a tidying of notation; it lays down the interface for the Kronecker product and the partial trace of §5.4—the higher-dimensional structures there can be expressed clearly only in this language. With the language of basic operations in place, §5.2 will show how partitioning into blocks completely rewrites the computational logic of multiplication.
:::

## 5.2 Multiplication of Block Matrices

Matrix multiplication $\mathbf{AB}$ admits four complementary geometric views, each revealing a different computational structure. Understanding these four views is the key both to understanding "why" general matrix multiplication is defined as it is and to extending multiplication to the language of blocks.

### 5.2.1 View One: Inner Products—Entry-by-Entry Computation

$$
(\mathbf{AB})_{ij} = \mathbf{row}_i(\mathbf{A}) \cdot \mathbf{col}_j(\mathbf{B}) = \sum_{k} a_{ik} b_{kj},
$$

where $\mathbf{row}_i(\mathbf{A})$ denotes row $i$ of $\mathbf{A}$ and $\mathbf{col}_j(\mathbf{B})$ denotes column $j$ of $\mathbf{B}$. Each entry of the result matrix is the inner product of a row of the left matrix with a column of the right matrix.

:::{prf:example} Inner-Product View
:label: ex-inner-product-view
Let

$$
\mathbf{A} = \begin{bmatrix} \textcolor{#FF8C00}{a} & \textcolor{#FF8C00}{b} \\ c & d \end{bmatrix},\quad
\mathbf{B} = \begin{bmatrix} \textcolor{#FF8C00}{1} & 2 \\ \textcolor{#FF8C00}{3} & 4 \end{bmatrix}.
$$

**Step 1.** Take row $0$ of $\mathbf{A}$, $\textcolor{#FF8C00}{\begin{bmatrix}a & b\end{bmatrix}}$, and column $0$ of $\mathbf{B}$, $\textcolor{#FF8C00}{\begin{bmatrix}1 \\ 3\end{bmatrix}}$.

**Step 2.** Compute the inner product $(\mathbf{AB})_{00} = \textcolor{#FF8C00}{a \cdot 1 + b \cdot 3} = a + 3b$.

**Step 3.** Repeat for every $(i,j)$, filling in the result matrix entry by entry:

$$
\mathbf{AB} = \begin{bmatrix} a+3b & 2a+4b \\ c+3d & 2c+4d \end{bmatrix}.
$$
:::

---

### 5.2.2 View Two: Linear Combinations of the Column Vectors of $\mathbf{A}$

Each column of $\mathbf{AB}$ is a linear combination of the column vectors of $\mathbf{A}$, with coefficients supplied by the corresponding column of $\mathbf{B}$:

$$
\mathbf{AB} = \mathbf{A}\begin{bmatrix} \mathbf{col}_0(\mathbf{B}) & \mathbf{col}_1(\mathbf{B}) & \cdots & \mathbf{col}_{n-1}(\mathbf{B}) \end{bmatrix}
= \begin{bmatrix} \mathbf{A}\,\mathbf{col}_0(\mathbf{B}) & \mathbf{A}\,\mathbf{col}_1(\mathbf{B}) & \cdots & \mathbf{A}\,\mathbf{col}_{n-1}(\mathbf{B}) \end{bmatrix}.
$$

In other words, column $j$ of $\mathbf{AB}$ is the linear combination of the columns of $\mathbf{A}$ whose coefficients are the components of $\mathbf{col}_j(\mathbf{B})$.

**Batched generalization (block view).** If the columns of $\mathbf{B}$ are partitioned into several blocks $[\mathbf{B}_0 \mid \mathbf{B}_1 \mid \cdots]$, then

$$
\mathbf{AB} = \begin{bmatrix} \mathbf{A}\mathbf{B}_0 & \mathbf{A}\mathbf{B}_1 & \cdots \end{bmatrix}.
$$

Each block $\mathbf{A}\mathbf{B}_k$ can be computed independently in parallel—this is precisely the algebraic basis of batched matrix operations on GPUs.

:::{prf:example} Column-Combination View
:label: ex-col-combination-view
Let

$$
\mathbf{A} = \begin{bmatrix} \textcolor{#006385}{a} & \textcolor{#FF8C00}{b} \\ \textcolor{#006385}{c} & \textcolor{#FF8C00}{d} \end{bmatrix},\quad
\mathbf{B} = \begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix}.
$$

**Step 1.** Column $0$ of $\mathbf{AB}$ $= \mathbf{A}\,\begin{bmatrix}1\\3\end{bmatrix} = 1\cdot\textcolor{#006385}{\begin{bmatrix}a\\c\end{bmatrix}} + 3\cdot\textcolor{#FF8C00}{\begin{bmatrix}b\\d\end{bmatrix}} = \begin{bmatrix}a+3b\\c+3d\end{bmatrix}$.

**Step 2.** Column $1$ of $\mathbf{AB}$ $= \mathbf{A}\,\begin{bmatrix}2\\4\end{bmatrix} = 2\cdot\textcolor{#006385}{\begin{bmatrix}a\\c\end{bmatrix}} + 4\cdot\textcolor{#FF8C00}{\begin{bmatrix}b\\d\end{bmatrix}} = \begin{bmatrix}2a+4b\\2c+4d\end{bmatrix}$.

**Step 3.** Combining the two, $\mathbf{AB} = \begin{bmatrix}a+3b & 2a+4b \\ c+3d & 2c+4d\end{bmatrix}$.
:::

---

### 5.2.3 View Three: Linear Combinations of the Row Vectors of $\mathbf{B}$

Symmetrically, each row of $\mathbf{AB}$ is a linear combination of the row vectors of $\mathbf{B}$, with coefficients supplied by the corresponding row of $\mathbf{A}$:

$$
\mathbf{AB} = \begin{bmatrix} \mathbf{row}_0(\mathbf{A}) \\ \mathbf{row}_1(\mathbf{A}) \\ \vdots \\ \mathbf{row}_{m-1}(\mathbf{A}) \end{bmatrix} \mathbf{B}
= \begin{bmatrix} \mathbf{row}_0(\mathbf{A})\,\mathbf{B} \\ \mathbf{row}_1(\mathbf{A})\,\mathbf{B} \\ \vdots \\ \mathbf{row}_{m-1}(\mathbf{A})\,\mathbf{B} \end{bmatrix}.
$$

In other words, row $i$ of $\mathbf{AB}$ is the linear combination of the rows of $\mathbf{B}$ whose coefficients are the components of $\mathbf{row}_i(\mathbf{A})$.

:::{prf:example} Row-Combination View
:label: ex-row-combination-view
Let

$$
\mathbf{A} = \begin{bmatrix} a & b \\ c & d \end{bmatrix},\quad
\mathbf{B} = \begin{bmatrix} \textcolor{#006385}{1} & \textcolor{#006385}{2} \\ \textcolor{#FF8C00}{3} & \textcolor{#FF8C00}{4} \end{bmatrix}.
$$

**Step 1.** Row $0$ of $\mathbf{AB}$ $= \begin{bmatrix}a & b\end{bmatrix}\mathbf{B} = a\cdot\textcolor{#006385}{\begin{bmatrix}1 & 2\end{bmatrix}} + b\cdot\textcolor{#FF8C00}{\begin{bmatrix}3 & 4\end{bmatrix}} = \begin{bmatrix}a+3b & 2a+4b\end{bmatrix}$.

**Step 2.** Row $1$ of $\mathbf{AB}$ $= \begin{bmatrix}c & d\end{bmatrix}\mathbf{B} = c\cdot\textcolor{#006385}{\begin{bmatrix}1 & 2\end{bmatrix}} + d\cdot\textcolor{#FF8C00}{\begin{bmatrix}3 & 4\end{bmatrix}} = \begin{bmatrix}c+3d & 2c+4d\end{bmatrix}$.

**Step 3.** Combining the two, $\mathbf{AB} = \begin{bmatrix}a+3b & 2a+4b \\ c+3d & 2c+4d\end{bmatrix}$.
:::

---

### 5.2.4 Block Matrix Multiplication

In the language of block matrices, the three views above are unified by a single formula.

:::{prf:theorem} Block Matrix Multiplication
:label: thm-block-multiplication

Let $\mathbf{A}$ be an $m\times p$ matrix and $\mathbf{B}$ a $p\times n$ matrix. Suppose the common dimension $p$ is partitioned at the points $0 = p_0 < p_1 < \cdots < p_s = p$, with the same partition applied to the columns of $\mathbf{A}$ and to the rows of $\mathbf{B}$, and denote the resulting sub-blocks by $\mathbf{A}_{ik}$ and $\mathbf{B}_{kj}$, respectively. Then

$$
(\mathbf{AB})_{ij} = \sum_{k=0}^{s-1} \mathbf{A}_{ik}\,\mathbf{B}_{kj}.
$$

This result is completely equivalent to ordinary matrix multiplication. **Dimension compatibility condition.** For each $k$, the number of columns of $\mathbf{A}_{ik}$ must equal the number of rows of $\mathbf{B}_{kj}$; that is, the partition points of the two matrices along the common dimension must agree.
:::


:::{prf:proof}
Let $r_a,c_b,p_k$ denote the partition offsets of the block rows of the result, the block columns of the result, and the inner dimension of the multiplication, respectively; the block indices are $a,b,k$, and the within-block indices are $u,v,t$.

1. Fix a valid $(u,v)$ within block $(a,b)$; its global position is $(r_a+u,c_b+v)$.
2. Group the sum over the inner dimension according to the common partition points $0=p_0<\cdots<p_s=p$:
$$
(\mathbf{AB})_{r_a+u,c_b+v}
=\sum_{k=0}^{s-1}\sum_{t=0}^{p_{k+1}-p_k-1}
a_{r_a+u,p_k+t}b_{p_k+t,c_b+v}
=\sum_{k=0}^{s-1}(\mathbf{A}_{ak}\mathbf{B}_{kb})_{uv}.
$$
3. Since the two sides agree at every position within the block, $(\mathbf{AB})_{ab}=\sum_{k=0}^{s-1}\mathbf{A}_{ak}\mathbf{B}_{kb}$, where the left-hand side denotes sub-block $(a,b)$.

$\square$
:::

:::{prf:example} Computing a Block Matrix Product
:label: ex-block-multiplication
Let

$$
\mathbf{A} = \begin{bmatrix} a & b & e \\ c & d & f \\ g & h & i \end{bmatrix}
= \begin{bmatrix} \mathbf{A}_{00} & \mathbf{A}_{01} \\ \mathbf{A}_{10} & \mathbf{A}_{11} \end{bmatrix},\quad
\mathbf{B} = \begin{bmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \\ 7 & 8 & 9 \end{bmatrix}
= \begin{bmatrix} \mathbf{B}_{00} & \mathbf{B}_{01} \\ \mathbf{B}_{10} & \mathbf{B}_{11} \end{bmatrix}.
$$

**Step 1.** Compute block $(0,0)$:

$$
(\mathbf{AB})_{00} = \mathbf{A}_{00}\mathbf{B}_{00} + \mathbf{A}_{01}\mathbf{B}_{10}
= \begin{bmatrix}a&b\\c&d\end{bmatrix}\begin{bmatrix}1&2\\4&5\end{bmatrix} + \begin{bmatrix}e\\f\end{bmatrix}\begin{bmatrix}7&8\end{bmatrix}
= \begin{bmatrix}a+4b+7e & 2a+5b+8e \\ c+4d+7f & 2c+5d+8f\end{bmatrix}.
$$

**Step 2.** Compute block $(0,1)$:

$$
(\mathbf{AB})_{01} = \mathbf{A}_{00}\mathbf{B}_{01} + \mathbf{A}_{01}\mathbf{B}_{11}
= \begin{bmatrix}a&b\\c&d\end{bmatrix}\begin{bmatrix}3\\6\end{bmatrix} + \begin{bmatrix}e\\f\end{bmatrix}\cdot 9
= \begin{bmatrix}3a+6b+9e \\ 3c+6d+9f\end{bmatrix}.
$$

**Step 3.** Compute blocks $(1,0)$ and $(1,1)$ in the same way, finally obtaining the $3\times 3$ matrix $\mathbf{AB}$.
:::

---

### 5.2.5 View Four: Outer-Product Expansion—Sums of Rank-One Matrices

$\mathbf{u}\mathbf{v}^* = |\mathbf{u}\rangle\langle\mathbf{v}|$ is called the **outer product** of $\mathbf{u}$ and $\mathbf{v}$; when both vectors are nonzero, it is a rank-one matrix.

The block multiplication theorem $(\mathbf{AB})_{ij} = \sum_k \mathbf{A}_{ik}\mathbf{B}_{kj}$ has a particularly intuitive geometric interpretation: when the blocks degenerate into single columns of the left factor and single rows of the right factor, each term $\mathbf{col}_k(\mathbf{A})\,\mathbf{row}_k(\mathbf{B})$ is a matrix of rank at most 1; when both vectors are nonzero its rank is 1, and the whole product is the sum of these rank-one matrices:

$$
\mathbf{AB} = \sum_{k} \mathbf{col}_k(\mathbf{A})\, \mathbf{row}_k(\mathbf{B}).
$$

Generalizing to the language of blocks, each term $\mathbf{A}_{ik}\mathbf{B}_{kj}$ is a "block-product contribution" whose rank need not be 1; summing them recovers the full product—this is precisely the geometric essence of the theorem.

:::{prf:example} Outer-Product Expansion View
:label: ex-outer-product-view
Let

$$
\mathbf{A} = \begin{bmatrix} \textcolor{#006385}{a} & \textcolor{#FF8C00}{b} \\ \textcolor{#006385}{c} & \textcolor{#FF8C00}{d} \end{bmatrix},\quad
\mathbf{B} = \begin{bmatrix} \textcolor{#006385}{1} & \textcolor{#006385}{2} \\ \textcolor{#FF8C00}{3} & \textcolor{#FF8C00}{4} \end{bmatrix}.
$$

**Step 1.** Take $k=0$: column $0$ of $\mathbf{A}$, $\textcolor{#006385}{\begin{bmatrix}a\\c\end{bmatrix}}$, and row $0$ of $\mathbf{B}$, $\textcolor{#006385}{\begin{bmatrix}1 & 2\end{bmatrix}}$; their outer product is a matrix of rank at most 1:

$$
\textcolor{#006385}{\begin{bmatrix}a\\c\end{bmatrix}\begin{bmatrix}1 & 2\end{bmatrix} = \begin{bmatrix}a & 2a \\ c & 2c\end{bmatrix}}.
$$

**Step 2.** Take $k=1$: column $1$ of $\mathbf{A}$, $\textcolor{#FF8C00}{\begin{bmatrix}b\\d\end{bmatrix}}$, and row $1$ of $\mathbf{B}$, $\textcolor{#FF8C00}{\begin{bmatrix}3 & 4\end{bmatrix}}$; their outer product is

$$
\textcolor{#FF8C00}{\begin{bmatrix}b\\d\end{bmatrix}\begin{bmatrix}3 & 4\end{bmatrix} = \begin{bmatrix}3b & 4b \\ 3d & 4d\end{bmatrix}}.
$$

**Step 3.** Add the two matrices of rank at most 1:

$$
\mathbf{AB} = \textcolor{#006385}{\begin{bmatrix}a & 2a \\ c & 2c\end{bmatrix}} + \textcolor{#FF8C00}{\begin{bmatrix}3b & 4b \\ 3d & 4d\end{bmatrix}} = \begin{bmatrix}a+3b & 2a+4b \\ c+3d & 2c+4d\end{bmatrix}.
$$

This agrees completely with the results of Views Two and Three.
:::

---

### 5.2.6 ◆ Strassen's Algorithm: The Power of Block Multiplication

A striking application of the block multiplication theorem is **Strassen's algorithm** (Strassen, 1969). Intuitively, multiplying two $2\times 2$ matrices takes $8$ scalar multiplications; Strassen discovered that $7$ suffice, by constructing the following $7$ intermediate quantities:

$$
\begin{aligned}
\mathbf{M}_0 &= (\mathbf{A}_{00}+\mathbf{A}_{11})(\mathbf{B}_{00}+\mathbf{B}_{11}),\\
\mathbf{M}_1 &= (\mathbf{A}_{10}+\mathbf{A}_{11})\mathbf{B}_{00},\\
\mathbf{M}_2 &= \mathbf{A}_{00}(\mathbf{B}_{01}-\mathbf{B}_{11}),\\
\mathbf{M}_3 &= \mathbf{A}_{11}(\mathbf{B}_{10}-\mathbf{B}_{00}),\\
\mathbf{M}_4 &= (\mathbf{A}_{00}+\mathbf{A}_{01})\mathbf{B}_{11},\\
\mathbf{M}_5 &= (\mathbf{A}_{10}-\mathbf{A}_{00})(\mathbf{B}_{00}+\mathbf{B}_{01}),\\
\mathbf{M}_6 &= (\mathbf{A}_{01}-\mathbf{A}_{11})(\mathbf{B}_{10}+\mathbf{B}_{11}),
\end{aligned}
$$

and then recombining them using additions:

$$
\mathbf{C}_{00} = \mathbf{M}_0+\mathbf{M}_3-\mathbf{M}_4+\mathbf{M}_6,\quad
\mathbf{C}_{01} = \mathbf{M}_2+\mathbf{M}_4,\quad
\mathbf{C}_{10} = \mathbf{M}_1+\mathbf{M}_3,\quad
\mathbf{C}_{11} = \mathbf{M}_0-\mathbf{M}_1+\mathbf{M}_2+\mathbf{M}_5.
$$

Applying this technique recursively to $n\times n$ matrices lowers the computational complexity from $O(n^3)$ to $O(n^{\log_2 7}) \approx O(n^{2.807})$.

:::{prf:remark} The Significance of Strassen's Algorithm
:label: rem-strassen

The importance of Strassen's algorithm lies not in saving a few multiplications but in the fact that it **shattered the intuitively "obvious" belief that "$n^3$ is a lower bound for matrix multiplication,"** and it opened up the research area of "lower bounds on the complexity of matrix multiplication," which is still developing today. Writing $\omega$ for the matrix multiplication exponent, which expresses the asymptotic complexity, it is known that $\omega\geq 2$; whether $\omega=2$ remains an open problem.
:::





::::{exercise} Four Views of Matrix Multiplication
:label: exer-ch5-four-views

Let $\mathbf{A} = \begin{bmatrix} 1 & 2 \\ 0 & 3 \end{bmatrix}$ and $\mathbf{B} = \begin{bmatrix} 2 & 1 \\ 1 & 4 \end{bmatrix}$.

**(a) (Inner-product view)** Compute $(\mathbf{AB})_{10}$ directly from the inner-product formula, that is, as the inner product of row $1$ of $\mathbf{A}$ with column $0$ of $\mathbf{B}$.

**(b) (Column-combination view)** Express column $1$ of $\mathbf{AB}$ as a linear combination of the columns of $\mathbf{A}$, and compute the result.

**(c) (Row-combination view)** Express row $0$ of $\mathbf{AB}$ as a linear combination of the rows of $\mathbf{B}$, and compute the result.

**(d) (Outer-product expansion view)** Write $\mathbf{AB}$ as the sum of two rank-one matrices, list each rank-one matrix separately, and then add them to check.

**(e)** Confirm that the four views give the same $\mathbf{AB}$.

:::{solution} exer-ch5-four-views
:class: dropdown

**(a)** $\mathbf{row}_1(\mathbf{A}) = \begin{bmatrix}0 & 3\end{bmatrix}$ and $\mathbf{col}_0(\mathbf{B}) = \begin{bmatrix}2\\1\end{bmatrix}$, so

$$(\mathbf{AB})_{10} = 0\cdot 2 + 3\cdot 1 = 3.$$

**(b)** Column $1$ of $\mathbf{B}$ is $\begin{bmatrix}1\\4\end{bmatrix}$, so

$$\text{column }1\text{ of }\mathbf{AB} = 1\cdot\begin{bmatrix}1\\0\end{bmatrix} + 4\cdot\begin{bmatrix}2\\3\end{bmatrix} = \begin{bmatrix}9\\12\end{bmatrix}.$$

**(c)** Row $0$ of $\mathbf{A}$ is $\begin{bmatrix}1 & 2\end{bmatrix}$, so

$$\text{row }0\text{ of }\mathbf{AB} = 1\cdot\begin{bmatrix}2 & 1\end{bmatrix} + 2\cdot\begin{bmatrix}1 & 4\end{bmatrix} = \begin{bmatrix}4 & 9\end{bmatrix}.$$

**(d)** The two rank-one matrices are

$$\begin{bmatrix}1\\0\end{bmatrix}\begin{bmatrix}2&1\end{bmatrix} = \begin{bmatrix}2&1\\0&0\end{bmatrix},\quad \begin{bmatrix}2\\3\end{bmatrix}\begin{bmatrix}1&4\end{bmatrix} = \begin{bmatrix}2&8\\3&12\end{bmatrix}.$$

Adding them gives

$$\mathbf{AB} = \begin{bmatrix}2&1\\0&0\end{bmatrix}+\begin{bmatrix}2&8\\3&12\end{bmatrix} = \begin{bmatrix}4&9\\3&12\end{bmatrix}.$$

**(e)** All four views give $\mathbf{AB} = \begin{bmatrix}4&9\\3&12\end{bmatrix}$, in agreement. ✓
:::
::::

---

::::{exercise} Dimension Compatibility Condition for Block Multiplication
:label: exer-ch5-block-compatibility

Let $\mathbf{A}$ be a $4\times 6$ matrix and $\mathbf{B}$ a $6\times 3$ matrix. Now partition the common dimension $6$ into the three segments $[0,2)$, $[2,5)$, and $[5,6)$.

**(a)** Write down the sizes of the sub-blocks $\mathbf{A}_{ik}$ ($k=0,1,2$) of $\mathbf{A}$ (assume that $\mathbf{A}$ is not partitioned along the row direction and is treated as a single block in that direction).

**(b)** Write down the sizes of the sub-blocks $\mathbf{B}_{kj}$ ($k=0,1,2$) of $\mathbf{B}$ (assume that $\mathbf{B}$ is not partitioned along the column direction).

**(c)** Verify that every pair $(\mathbf{A}_{ik},\mathbf{B}_{kj})$ satisfies the dimension compatibility condition, and state the size of the three matrices whose sum is $(\mathbf{AB})_{00}$.

:::{solution} exer-ch5-block-compatibility
:class: dropdown

**(a)** The partition points divide the common dimension into widths $2,3,1$, and $\mathbf{A}$ is a single block along the row direction ($4$ rows), so

$$\mathbf{A}_{i0}: 4\times 2,\quad \mathbf{A}_{i1}: 4\times 3,\quad \mathbf{A}_{i2}: 4\times 1.$$

**(b)** The same partition points determine how $\mathbf{B}$ is segmented along the row direction, and $\mathbf{B}$ is a single block along the column direction ($3$ columns), so

$$\mathbf{B}_{0j}: 2\times 3,\quad \mathbf{B}_{1j}: 3\times 3,\quad \mathbf{B}_{2j}: 1\times 3.$$

**(c)** For each $k$, the number of columns of $\mathbf{A}_{ik}$ equals the number of rows of $\mathbf{B}_{kj}$: $2=2$, $3=3$, and $1=1$, respectively, so the dimensions are compatible. $(\mathbf{AB})_{00}$ is the sum of three $4\times 3$ matrices and is therefore a $4\times 3$ matrix.
:::
::::

---

::::{exercise} Rank Properties of Outer-Product Decompositions
:label: exer-ch5-rank-outer

Let $\mathbf{u}\in\mathbb{R}^m$ and $\mathbf{v}^\top\in\mathbb{R}^{1\times n}$ both be nonzero vectors.

**(a)** Explain why the rank of $\mathbf{u}\mathbf{v}^\top$ must be $1$. (Hint: consider its column space.)

**(b)** Let $\mathbf{A}=\begin{bmatrix}2\\1\end{bmatrix}\begin{bmatrix}1&3\end{bmatrix}$. Without expanding, determine the rank of $\mathbf{A}$ directly, and find a nonzero vector $\mathbf{x}$ such that $\mathbf{Ax}=\mathbf{0}$.

**(c)** The outer-product expansion formula $\mathbf{AB}=\sum_k \mathbf{col}_k(\mathbf{A})\,\mathbf{row}_k(\mathbf{B})$ is a sum of $p$ matrices of rank at most 1 (a term in which $\mathbf{col}_k(\mathbf{A})$ and $\mathbf{row}_k(\mathbf{B})$ are both nonzero has rank 1). At most how large can the rank of this sum be? If every term is nonzero, how small can it be? Give an example of each.

:::{solution} exer-ch5-rank-outer
:class: dropdown

**(a)** Every column of $\mathbf{u}\mathbf{v}^\top$ is a scalar multiple of $\mathbf{u}$, so the column space is at most one-dimensional, that is, $\mathrm{rank}(\mathbf{u}\mathbf{v}^\top)\leq 1$. Moreover, since $\mathbf{u},\mathbf{v}$ are nonzero, there are components whose product is nonzero, so the rank is exactly $1$.

**(b)** $\mathrm{rank}(\mathbf{A})=1$. Every column is a multiple of $\begin{bmatrix}2\\1\end{bmatrix}$, and $\mathbf{A}\mathbf{x}=\mathbf{0}$ if and only if $\mathbf{v}^\top\mathbf{x}=0$, that is, $x_0+3x_1=0$; take $\mathbf{x}=\begin{bmatrix}3\\-1\end{bmatrix}$.

**(c)** It is at most $\min(p,m,n)$ (the rank of a sum of $p$ rank-one matrices exceeds neither $p$ nor the limit imposed by the size of the matrix itself). It can be as small as $0$, but this requires at least two terms to cancel. Take $p=2$ and let

$$
\mathbf{A}=\begin{bmatrix}\mathbf{u} & -\mathbf{u}\end{bmatrix}=\begin{bmatrix}1&-1\\-1&1\end{bmatrix},\qquad
\mathbf{B}=\begin{bmatrix}\mathbf{v}^\top\\ \mathbf{v}^\top\end{bmatrix}=\begin{bmatrix}1&1\\1&1\end{bmatrix},
$$

where $\mathbf{u}=\begin{bmatrix}1\\-1\end{bmatrix}$ and $\mathbf{v}=\begin{bmatrix}1\\1\end{bmatrix}$. Expanding by the outer-product expansion formula,

$$
\mathbf{AB}=\mathbf{col}_0(\mathbf{A})\,\mathbf{row}_0(\mathbf{B})+\mathbf{col}_1(\mathbf{A})\,\mathbf{row}_1(\mathbf{B})
=\mathbf{u}\mathbf{v}^\top+(-\mathbf{u})\mathbf{v}^\top=\mathbf{0},
$$

so the two rank-one outer products cancel exactly, and the rank of the sum is $0$. (Note: with a single term, $p=1$, a nonzero outer product always has rank $1$ and cannot reach $0$.)
:::
::::

---

::::{exercise} Verifying Strassen's Formulas
:label: exer-ch5-strassen-verify

Let $\mathbf{A}=\begin{bmatrix}1&2\\3&4\end{bmatrix}$ and $\mathbf{B}=\begin{bmatrix}5&6\\7&8\end{bmatrix}$, regarded as $2\times 2$ block matrices with $1\times 1$ sub-blocks (that is, every sub-block is a scalar).

**(a)** Compute Strassen's seven intermediate quantities $M_0,\ldots,M_6$.

**(b)** Use the recombination formulas to compute $\mathbf{C}_{00},\mathbf{C}_{01},\mathbf{C}_{10},\mathbf{C}_{11}$, and compare with the result of direct multiplication.

**(c)** In the scalar setting of this exercise, how many multiplications does direct multiplication require in total? How many does Strassen's algorithm require? How many are saved?

:::{solution} exer-ch5-strassen-verify
:class: dropdown

First carry out direct multiplication: $\mathbf{AB}=\begin{bmatrix}19&22\\43&50\end{bmatrix}$.

**(a)** With $A_{00}=1,A_{01}=2,A_{10}=3,A_{11}=4$ and $B_{00}=5,B_{01}=6,B_{10}=7,B_{11}=8$:

$$\begin{aligned}
M_0&=(1+4)(5+8)=5\times13=65,\\
M_1&=(3+4)\times5=7\times5=35,\\
M_2&=1\times(6-8)=1\times(-2)=-2,\\
M_3&=4\times(7-5)=4\times2=8,\\
M_4&=(1+2)\times8=3\times8=24,\\
M_5&=(3-1)(5+6)=2\times11=22,\\
M_6&=(2-4)(7+8)=(-2)\times15=-30.
\end{aligned}$$

**(b)**

$$\begin{aligned}
C_{00}&=M_0+M_3-M_4+M_6=65+8-24+(-30)=19,\\
C_{01}&=M_2+M_4=-2+24=22,\\
C_{10}&=M_1+M_3=35+8=43,\\
C_{11}&=M_0-M_1+M_2+M_5=65-35+(-2)+22=50.
\end{aligned}$$

This agrees exactly with $\mathbf{AB}=\begin{bmatrix}19&22\\43&50\end{bmatrix}$. ✓

**(c)** Direct multiplication: $8$; Strassen: $7$; saved: $1$. (The asymptotic advantage appears only when the method is applied recursively to large matrices.)
:::
::::

### 5.2.7 Python: Block Matrix Multiplication with NumPy

In [ ]:
import numpy as np

print("Comparing block matrix multiplication with 4D tensors")
print("=" * 40)

# Create block matrices in 4D tensor form
# A: (2, 3, 4, 4) - 2x3 blocks, each a 4x4 matrix
# B: (3, 2, 4, 4) - 3x2 blocks, each a 4x4 matrix
np.random.seed(42)
A_4d = np.random.randn(2, 3, 4, 4)
B_4d = np.random.randn(3, 2, 4, 4)

print(f"Shape of A as a 4D tensor: {A_4d.shape}")
print(f"Shape of B as a 4D tensor: {B_4d.shape}")

print("\n" + "-" * 40)
print("Method 1: convert back to 2D matrices, then multiply as usual")
print("-" * 40)

def tensor_4d_to_2d(tensor_4d):
    """Convert a 4D block tensor into a 2D matrix"""
    num_block_rows, num_block_cols, block_height, block_width = tensor_4d.shape
    
    # Compute the size of the final matrix
    total_height = num_block_rows * block_height
    total_width = num_block_cols * block_width
    
    # Initialize the result matrix
    matrix_2d = np.zeros((total_height, total_width), dtype=tensor_4d.dtype)
    
    # Fill in each block
    for i in range(num_block_rows):
        for j in range(num_block_cols):
            row_start = i * block_height
            row_end = (i + 1) * block_height
            col_start = j * block_width
            col_end = (j + 1) * block_width
            
            matrix_2d[row_start:row_end, col_start:col_end] = tensor_4d[i, j]
    
    return matrix_2d

# Convert the 4D tensors into 2D matrices
A_2d = tensor_4d_to_2d(A_4d)
B_2d = tensor_4d_to_2d(B_4d)

print(f"Shape of A as a 2D matrix: {A_2d.shape}")
print(f"Shape of B as a 2D matrix: {B_2d.shape}")

# Ordinary matrix multiplication
C_2d_method1 = A_2d @ B_2d
print(f"Shape of the method 1 result: {C_2d_method1.shape}")

print("\n" + "-" * 40)
print("Method 2: multiply the 4D tensors with einsum, then convert back")
print("-" * 40)

# Block matrix multiplication with einsum
# 'ikmn,klno->ilmo': k and n are the summation indices
# i,l: block-row and block-column indices of the result; m,o: within-block row and column indices
C_4d = np.einsum('ikmn,klno->ilmo', A_4d, B_4d)

print(f"Shape of the 4D einsum result: {C_4d.shape}")

# Convert the result into a 2D matrix
C_2d_method2 = tensor_4d_to_2d(C_4d)
print(f"Shape of the method 2 result: {C_2d_method2.shape}")

print("\n" + "-" * 40)
print("Comparing the results")
print("-" * 40)

# Compare the results of the two methods
difference = np.abs(C_2d_method1 - C_2d_method2)
max_diff = np.max(difference)
mean_diff = np.mean(difference)

print(f"The two methods agree: {np.allclose(C_2d_method1, C_2d_method2)}")
print(f"Maximum difference: {max_diff:.2e}")
print(f"Mean difference: {mean_diff:.2e}")

print("\n" + "-" * 40)
print("Detailed check: computing one block by hand")
print("-" * 40)

# Compute block (0,0) of the result by hand as a check
# C[0,0] = A[0,0]@B[0,0] + A[0,1]@B[1,0] + A[0,2]@B[2,0]
manual_block_00 = (A_4d[0,0] @ B_4d[0,0] + 
                   A_4d[0,1] @ B_4d[1,0] + 
                   A_4d[0,2] @ B_4d[2,0])

# Extract the corresponding block from the method 1 result
method1_block_00 = C_2d_method1[0:4, 0:4]

# Extract the corresponding block from the method 2 result  
method2_block_00 = C_2d_method2[0:4, 0:4]

print("By hand vs method 1 vs method 2:")
print(f"By hand equals method 1: {np.allclose(manual_block_00, method1_block_00)}")
print(f"By hand equals method 2: {np.allclose(manual_block_00, method2_block_00)}")
print(f"Method 1 equals method 2: {np.allclose(method1_block_00, method2_block_00)}")

:::{admonition} §5.2 Summary
:class: tip
This section answers what the "meaning" of the matrix product $\mathbf{AB}$ is, and how partitioning into blocks rewrites its computational logic. Multiplication admits four complementary geometric intuitions: the inner-product view understands each entry of the result as the dot product of a row vector of $\mathbf{A}$ with a column vector of $\mathbf{B}$; the column-combination view regards each column of $\mathbf{AB}$ as a linear combination of the columns of $\mathbf{A}$, while the row-combination view regards each row as a linear combination of the rows of $\mathbf{B}$, connecting multiplication directly with the language of linear spaces; and the outer-product expansion view decomposes the whole product into a sum of matrices of rank at most 1, $\sum_k \mathrm{col}_k(\mathbf{A})\,\mathrm{row}_k(\mathbf{B})$, each term revealing the independent contribution of one column–row pair of vectors.

Generalized to block matrices, the four views come down to a single core formula, $(\mathbf{AB})_{ij} = \sum_k \mathbf{A}_{ik}\mathbf{B}_{kj}$, provided that the dimensions of adjacent blocks match. Strassen's algorithm was born within exactly this framework: it completes a $2\times 2$ block product with 7 multiplications, shattering the belief in $n^3$ complexity. The next section builds on multiplication to discuss inversion and introduces the most important tool of this chapter—the Schur complement.
:::

## 5.3 Inverses of Matrices in Block Form

For matrices with certain special structures (such as diagonal, upper triangular, or lower triangular matrices), we can use the block structure to invert in stages and reduce the computational complexity. In the partitions considered here, every block on the diagonal is required to be a square matrix.

### 5.3.1 Inverses of Block Diagonal Matrices

::: {prf:definition} Block Diagonal Matrix
:label: def-block-diagonal
A block diagonal matrix is a special block matrix whose off-diagonal blocks are all zero matrices:

$$\mathbf{D} = \text{diag}(\mathbf{D}_0, \mathbf{D}_1, \ldots, \mathbf{D}_{k-1}) = 
\begin{bmatrix}
\mathbf{D}_0 & \mathbf{0} & \cdots & \mathbf{0} \\
\mathbf{0} & \mathbf{D}_1 & \cdots & \mathbf{0} \\
\vdots & \vdots & \ddots & \vdots \\
\mathbf{0} & \mathbf{0} & \cdots & \mathbf{D}_{k-1}
\end{bmatrix}$$
:::

::: {prf:property} Properties of Operations on Block Diagonal Matrices
:label: pro-block-diagonal
For a block diagonal matrix $\mathbf{D} = \text{diag}(\mathbf{D}_0, \mathbf{D}_1, \ldots, \mathbf{D}_{k-1})$, the following properties hold:

1. $\mathbf{D}^n = \text{diag}(\mathbf{D}_0^n, \mathbf{D}_1^n, \ldots, \mathbf{D}_{k-1}^n)$
2. $\det(\mathbf{D}) = \det(\mathbf{D}_0) \cdot \det(\mathbf{D}_1) \cdots \det(\mathbf{D}_{k-1})$ (a rigorous proof of this property relies on the theory of determinants; see Chapter 7)
3. $\text{tr}(\mathbf{D}) = \text{tr}(\mathbf{D}_0) + \text{tr}(\mathbf{D}_1) + \cdots + \text{tr}(\mathbf{D}_{k-1})$
4. If every diagonal block is invertible, then $\mathbf{D}^{-1} = \text{diag}(\mathbf{D}_0^{-1}, \mathbf{D}_1^{-1}, \ldots, \mathbf{D}_{k-1}^{-1})$
:::

:::{prf:remark} The Geometric Meaning of Block Diagonal Structure: Direct Sum Decompositions
:label: rem-block-diag-direct-sum

A block diagonal matrix is not just a coincidence in which "the off-diagonal blocks happen to be zero"; it has a definite geometric origin. If a space decomposes as a direct sum of subspaces $V=V_0\oplus V_1\oplus\cdots\oplus V_{k-1}$ and a linear mapping maps each $V_i$ into itself (that is, each $V_i$ is an **invariant subspace**), then with respect to the basis assembled according to the direct sum, the matrix of the mapping is exactly block diagonal—each diagonal block $\mathbf{D}_i$ is the representation of the mapping restricted to $V_i$. The direct sum decomposition established in the Chapter 4 exercise {ref}`exer-ch4-direct-sum` is precisely the geometric root of why "inverting block by block and taking powers block by block" works here: the blocks do not interfere with one another because the subspaces do not interfere with one another.
:::

These properties make computations with block diagonal matrices very efficient, because we can operate on each diagonal block separately and then combine the results.

::: {prf:observation} Inverses of Block Diagonal Matrices
:label: obs-block-diag-inverse
To invert a block diagonal matrix, one need only invert each diagonal block separately; the off-diagonal blocks remain zero.
:::


---

### 5.3.2 The Block Formula for the Inverse and the Schur Complement

For a general $2\times 2$ block matrix

$$
\mathbf{M} = \begin{pmatrix} \mathbf{A} & \mathbf{B} \\ \mathbf{C} & \mathbf{D} \end{pmatrix},
$$

where $\mathbf{A}$ is a $p\times p$ square matrix and $\mathbf{D}$ is a $q\times q$ square matrix, we would like to follow the strategy of §5.3.1 and reduce the computation of $\mathbf{M}^{-1}$ to inverting smaller matrices. The difficulty is that the off-diagonal blocks $\mathbf{B}$ and $\mathbf{C}$ couple the blocks to one another—the way out is to first use block multiplication to "eliminate" one of the off-diagonal blocks.

**Derivation: block Gaussian elimination.** Assume that $\mathbf{A}$ is invertible. Imitating the step in ordinary Gaussian elimination of "using row 0 to eliminate the entries below it," we left-multiply by a block lower triangular matrix to eliminate the block $\mathbf{C}$ in the lower-left corner of $\mathbf{M}$:

$$
\begin{pmatrix} \mathbf{I} & \mathbf{0} \\ -\mathbf{C}\mathbf{A}^{-1} & \mathbf{I} \end{pmatrix}
\begin{pmatrix} \mathbf{A} & \mathbf{B} \\ \mathbf{C} & \mathbf{D} \end{pmatrix}
=
\begin{pmatrix} \mathbf{A} & \mathbf{B} \\ \mathbf{C}-\mathbf{C}\mathbf{A}^{-1}\mathbf{A} & \mathbf{D}-\mathbf{C}\mathbf{A}^{-1}\mathbf{B} \end{pmatrix}
=
\begin{pmatrix} \mathbf{A} & \mathbf{B} \\ \mathbf{0} & \mathbf{S} \end{pmatrix},
$$

where the matrix that appears naturally in the lower-right corner,

$$
\mathbf{S} = \mathbf{D} - \mathbf{C}\,\mathbf{A}^{-1}\mathbf{B}
$$

is called the **Schur complement** of $\mathbf{A}$ in $\mathbf{M}$. What remains after the elimination is a block upper triangular matrix, and by {prf:ref}`pro-block-triangular-invertible`, to be established in §5.3.3 (a block triangular matrix is invertible $\leftrightarrow$ each diagonal block is invertible), as long as $\mathbf{S}$ is also invertible, the whole matrix $\mathbf{M}$ is invertible.

:::{prf:theorem} Block Inversion Formula
:label: thm-block-inverse-schur

Let $\mathbf{M} = \begin{pmatrix} \mathbf{A} & \mathbf{B} \\ \mathbf{C} & \mathbf{D} \end{pmatrix}$. If $\mathbf{A}$ and the Schur complement $\mathbf{S} = \mathbf{D} - \mathbf{C}\mathbf{A}^{-1}\mathbf{B}$ are both invertible, then $\mathbf{M}$ is invertible, and

$$
\mathbf{M}^{-1} = \begin{pmatrix}
\mathbf{A}^{-1} + \mathbf{A}^{-1}\mathbf{B}\,\mathbf{S}^{-1}\mathbf{C}\,\mathbf{A}^{-1} & -\mathbf{A}^{-1}\mathbf{B}\,\mathbf{S}^{-1} \\
-\mathbf{S}^{-1}\mathbf{C}\,\mathbf{A}^{-1} & \mathbf{S}^{-1}
\end{pmatrix}.
$$
:::

:::{prf:proof}
Denote the right-hand side of the formula by $\mathbf{N}$, and verify $\mathbf{M}\mathbf{N}=\mathbf{I}$ by multiplying out directly with the block multiplication theorem ({prf:ref}`thm-block-multiplication`). Computing block by block:

1. Upper-left block: $\mathbf{A}\left(\mathbf{A}^{-1} + \mathbf{A}^{-1}\mathbf{B}\mathbf{S}^{-1}\mathbf{C}\mathbf{A}^{-1}\right) + \mathbf{B}\left(-\mathbf{S}^{-1}\mathbf{C}\mathbf{A}^{-1}\right) = \mathbf{I} + \mathbf{B}\mathbf{S}^{-1}\mathbf{C}\mathbf{A}^{-1} - \mathbf{B}\mathbf{S}^{-1}\mathbf{C}\mathbf{A}^{-1} = \mathbf{I}$.
2. Upper-right block: $\mathbf{A}\left(-\mathbf{A}^{-1}\mathbf{B}\mathbf{S}^{-1}\right) + \mathbf{B}\,\mathbf{S}^{-1} = -\mathbf{B}\mathbf{S}^{-1} + \mathbf{B}\mathbf{S}^{-1} = \mathbf{0}$.
3. Lower-left block: $\mathbf{C}\left(\mathbf{A}^{-1} + \mathbf{A}^{-1}\mathbf{B}\mathbf{S}^{-1}\mathbf{C}\mathbf{A}^{-1}\right) + \mathbf{D}\left(-\mathbf{S}^{-1}\mathbf{C}\mathbf{A}^{-1}\right) = \mathbf{C}\mathbf{A}^{-1} + (\mathbf{C}\mathbf{A}^{-1}\mathbf{B} - \mathbf{D})\mathbf{S}^{-1}\mathbf{C}\mathbf{A}^{-1} = \mathbf{C}\mathbf{A}^{-1} - \mathbf{S}\,\mathbf{S}^{-1}\mathbf{C}\mathbf{A}^{-1} = \mathbf{0}$.
4. Lower-right block: $\mathbf{C}\left(-\mathbf{A}^{-1}\mathbf{B}\mathbf{S}^{-1}\right) + \mathbf{D}\,\mathbf{S}^{-1} = (\mathbf{D} - \mathbf{C}\mathbf{A}^{-1}\mathbf{B})\mathbf{S}^{-1} = \mathbf{S}\mathbf{S}^{-1} = \mathbf{I}$.

In the same way one verifies $\mathbf{N}\mathbf{M}=\mathbf{I}$, so $\mathbf{M}^{-1}=\mathbf{N}$. $\square$
:::

Note the cancellation $\mathbf{C}\mathbf{A}^{-1}\mathbf{B} - \mathbf{D} = -\mathbf{S}$ in the computation of the lower-left block—the Schur complement is precisely the "recipe" that makes all the cross terms cancel exactly. This is no coincidence; it is the trace that block Gaussian elimination leaves behind at the level of the formula.

:::{prf:example} Computing the Inverse of a $4\times 4$ Matrix with the Schur Complement
:label: ex-schur-inverse-4x4

Let

$$
\mathbf{M} = \begin{pmatrix}
2 & 1 & 1 & 0 \\
1 & 1 & 0 & 1 \\
1 & 2 & 3 & 1 \\
0 & 1 & 1 & 2
\end{pmatrix}
= \begin{pmatrix} \mathbf{A} & \mathbf{B} \\ \mathbf{C} & \mathbf{D} \end{pmatrix},
\quad
\mathbf{A} = \begin{pmatrix} 2 & 1 \\ 1 & 1 \end{pmatrix},\;
\mathbf{B} = \begin{pmatrix} 1 & 0 \\ 0 & 1 \end{pmatrix},\;
\mathbf{C} = \begin{pmatrix} 1 & 2 \\ 0 & 1 \end{pmatrix},\;
\mathbf{D} = \begin{pmatrix} 3 & 1 \\ 1 & 2 \end{pmatrix}.
$$

**Step 1.** Find $\mathbf{A}^{-1}$. Since $\det\mathbf{A} = 2\cdot 1 - 1\cdot 1 = 1$, we have $\mathbf{A}^{-1} = \begin{pmatrix} 1 & -1 \\ -1 & 2 \end{pmatrix}$.

**Step 2.** Compute the Schur complement. First compute $\mathbf{C}\mathbf{A}^{-1} = \begin{pmatrix} 1 & 2 \\ 0 & 1 \end{pmatrix}\begin{pmatrix} 1 & -1 \\ -1 & 2 \end{pmatrix} = \begin{pmatrix} -1 & 3 \\ -1 & 2 \end{pmatrix}$; since $\mathbf{B}=\mathbf{I}$, we obtain

$$
\mathbf{S} = \mathbf{D} - \mathbf{C}\mathbf{A}^{-1}\mathbf{B}
= \begin{pmatrix} 3 & 1 \\ 1 & 2 \end{pmatrix} - \begin{pmatrix} -1 & 3 \\ -1 & 2 \end{pmatrix}
= \begin{pmatrix} 4 & -2 \\ 2 & 0 \end{pmatrix},
\qquad
\mathbf{S}^{-1} = \frac{1}{4}\begin{pmatrix} 0 & 2 \\ -2 & 4 \end{pmatrix}
= \begin{pmatrix} 0 & \tfrac{1}{2} \\ -\tfrac{1}{2} & 1 \end{pmatrix}.
$$

**Step 3.** Substitute into the block inversion formula and compute block by block (simplifying with $\mathbf{B}=\mathbf{I}$):

$$
\mathbf{M}^{-1} = \begin{pmatrix}
\mathbf{A}^{-1} + \mathbf{A}^{-1}\mathbf{S}^{-1}\mathbf{C}\mathbf{A}^{-1} & -\mathbf{A}^{-1}\mathbf{S}^{-1} \\
-\mathbf{S}^{-1}\mathbf{C}\mathbf{A}^{-1} & \mathbf{S}^{-1}
\end{pmatrix}
= \begin{pmatrix}
1 & -\tfrac{1}{2} & -\tfrac{1}{2} & \tfrac{1}{2} \\
-\tfrac{3}{2} & 2 & 1 & -\tfrac{3}{2} \\
\tfrac{1}{2} & -1 & 0 & \tfrac{1}{2} \\
\tfrac{1}{2} & -\tfrac{1}{2} & -\tfrac{1}{2} & 1
\end{pmatrix}.
$$

**Step 4.** Verification. One can check $\mathbf{M}\mathbf{M}^{-1} = \mathbf{I}_4$ block by block or directly (see the code check below). The original inversion of a $4\times 4$ matrix has been broken down into two $2\times 2$ inversions ($\mathbf{A}$ and $\mathbf{S}$) plus a number of $2\times 2$ multiplications.
:::

:::{prf:remark} Schur's Determinant Formula
:label: rem-schur-determinant

Schur's original result of 1917 was precisely about determinants: under the condition that $\mathbf{A}$ is invertible,

$$
\det \mathbf{M} = \det \mathbf{A} \cdot \det \mathbf{S} = \det \mathbf{A} \cdot \det\!\left(\mathbf{D} - \mathbf{C}\mathbf{A}^{-1}\mathbf{B}\right).
$$

Intuitively, block Gaussian elimination reduces $\mathbf{M}$ to the block upper triangular matrix $\begin{pmatrix}\mathbf{A} & \mathbf{B} \\ \mathbf{0} & \mathbf{S}\end{pmatrix}$; the elimination step does not change the determinant, and the determinant of a block triangular matrix equals the product of the determinants of its diagonal blocks. Checking with the example above: $\det\mathbf{M} = \det\mathbf{A}\cdot\det\mathbf{S} = 1 \times 4 = 4$. These facts about determinants will be established rigorously in the theory of determinants in Chapter 7, at which point this formula will follow naturally.
:::

::::{exercise} Practice with Schur Complement Inversion
:label: exer-ch5-schur-inverse

Let $\mathbf{M} = \begin{pmatrix} \mathbf{I} & \mathbf{I} \\ \mathbf{C} & \mathbf{D} \end{pmatrix}$, where $\mathbf{I}$ is the $2\times 2$ identity matrix, $\mathbf{C} = \begin{pmatrix} 1 & 1 \\ 0 & 1 \end{pmatrix}$, and $\mathbf{D} = \begin{pmatrix} 3 & 1 \\ 1 & 2 \end{pmatrix}$.

**(a)** Compute the Schur complement $\mathbf{S}$ and $\det\mathbf{M}$.

**(b)** Use the block inversion formula to compute $\mathbf{M}^{-1}$, and check it against $\mathbf{M}\mathbf{M}^{-1}=\mathbf{I}_4$.

:::{solution} exer-ch5-schur-inverse
:class: dropdown

**(a)** Since $\mathbf{A}=\mathbf{B}=\mathbf{I}$, the Schur complement simplifies to

$$
\mathbf{S} = \mathbf{D} - \mathbf{C}\mathbf{I}^{-1}\mathbf{I} = \mathbf{D} - \mathbf{C} = \begin{pmatrix} 2 & 0 \\ 1 & 1 \end{pmatrix},
$$

and by {prf:ref}`rem-schur-determinant`, $\det\mathbf{M} = \det\mathbf{I}\cdot\det\mathbf{S} = 1\times 2 = 2$.

**(b)** $\mathbf{S}^{-1} = \frac{1}{2}\begin{pmatrix} 1 & 0 \\ -1 & 2 \end{pmatrix} = \begin{pmatrix} \tfrac{1}{2} & 0 \\ -\tfrac{1}{2} & 1 \end{pmatrix}$. Substituting into the formula ($\mathbf{A}^{-1}=\mathbf{B}=\mathbf{I}$):

$$
\mathbf{M}^{-1} = \begin{pmatrix}
\mathbf{I} + \mathbf{S}^{-1}\mathbf{C} & -\mathbf{S}^{-1} \\
-\mathbf{S}^{-1}\mathbf{C} & \mathbf{S}^{-1}
\end{pmatrix}
= \begin{pmatrix}
\tfrac{3}{2} & \tfrac{1}{2} & -\tfrac{1}{2} & 0 \\
-\tfrac{1}{2} & \tfrac{3}{2} & \tfrac{1}{2} & -1 \\
-\tfrac{1}{2} & -\tfrac{1}{2} & \tfrac{1}{2} & 0 \\
\tfrac{1}{2} & -\tfrac{1}{2} & -\tfrac{1}{2} & 1
\end{pmatrix},
$$

where $\mathbf{S}^{-1}\mathbf{C} = \begin{pmatrix} \tfrac{1}{2} & \tfrac{1}{2} \\ -\tfrac{1}{2} & \tfrac{1}{2} \end{pmatrix}$. Multiplying out directly verifies $\mathbf{M}\mathbf{M}^{-1}=\mathbf{I}_4$. ✓
:::
::::

**Summary.**  
The Schur complement is not a formula to be memorized; it is the natural product of block Gaussian elimination. The residue left in the lower-right corner after $\mathbf{C}$ has been eliminated measures exactly "how much invertibility $\mathbf{D}$ retains once the coupling transmitted through $\mathbf{A}$ has been subtracted." Through the block formula, inverting a large matrix is reduced to inverting two smaller matrices ($\mathbf{A}$ and $\mathbf{S}$) plus matrix multiplications—this is the embryonic form of the block elimination and LU decomposition of Chapter 6.

---


In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True, linewidth=100)

print("=" * 60)
print("5.3.2 Schur complement block inversion formula - numerical check")
print("=" * 60)

# The four sub-blocks from the example
A = np.array([[2., 1.], [1., 1.]])
B = np.eye(2)
C = np.array([[1., 2.], [0., 1.]])
D = np.array([[3., 1.], [1., 2.]])

M = np.block([[A, B], [C, D]])
print("Block matrix M =")
print(M)

# Step 1: Schur complement S = D - C A⁻¹ B
A_inv = np.linalg.inv(A)
S = D - C @ A_inv @ B
print("\nSchur complement S = D - C A⁻¹ B =")
print(S)

# Step 2: block inversion formula
S_inv = np.linalg.inv(S)
M_inv_block = np.block([
    [A_inv + A_inv @ B @ S_inv @ C @ A_inv, -A_inv @ B @ S_inv],
    [-S_inv @ C @ A_inv,                     S_inv            ]
])
print("\nM⁻¹ from the block formula =")
print(M_inv_block)

# Step 3: compare with direct inversion in NumPy
M_inv_direct = np.linalg.inv(M)
print("\nnp.linalg.inv(M), direct inversion =")
print(M_inv_direct)
print("\nThe two agree:", np.allclose(M_inv_block, M_inv_direct))
print("M @ M⁻¹ = I:", np.allclose(M @ M_inv_block, np.eye(4)))

# Step 4: Schur determinant formula det M = det A · det S
print("\ndet(M)          =", np.linalg.det(M).round(4))
print("det(A) * det(S) =", (np.linalg.det(A) * np.linalg.det(S)).round(4))


### 5.3.3 Inverses of Upper and Lower Triangular Matrices

Triangular matrices are an important class of matrices with special structure, and both the computation and the properties of their inverses have distinctive features.

**Basic Definitions**

:::{prf:definition} Upper and Lower Triangular Matrices
:label: def-triangular-matrices
For an $m \times n$ matrix $\mathbf{U}$:
- if $u_{ij} = 0$ (whenever $i > j$), then $\mathbf{U}$ is called an **upper triangular matrix**; that is, all entries below the main diagonal are zero
- if $u_{ij} = 0$ (whenever $i < j$), then $\mathbf{U}$ is called a **lower triangular matrix**; that is, all entries above the main diagonal are zero
:::

:::{prf:definition} Unit Triangular Matrices
:label: def-unit-triangular-matrices
- If the diagonal entries of an upper triangular matrix $\mathbf{U}$ satisfy $u_{ii} = 1$, it is called a **unit upper triangular matrix**
- If the diagonal entries of a lower triangular matrix $\mathbf{L}$ satisfy $l_{ii} = 1$, it is called a **unit lower triangular matrix**
:::

**Basic Properties of Triangular Matrices**

Triangular matrices are well behaved under multiplication: the product of two upper triangular matrices is again upper triangular, and the product of two lower triangular matrices is again lower triangular.

**Inverses of Block Triangular Matrices**



Continuing the block matrix inversion of the previous section, a square block matrix with triangular structure has a comparatively simple inverse, which likewise preserves the triangular structure:

:::{prf:example} Inverse of a Square Block Upper Triangular Matrix
:label: ex-triangular-block-inverse
Let the block upper triangular matrix be (with the diagonal blocks $\mathbf{U}_{00}$ and $\mathbf{U}_{11}$ invertible square matrices)

$$\mathbf{U} = \begin{bmatrix} \mathbf{U}_{00} & \mathbf{U}_{01} \\ \mathbf{0} & \mathbf{U}_{11} \end{bmatrix}$$  
Then its inverse is

$$\mathbf{U}^{-1} = \begin{bmatrix} \mathbf{U}_{00}^{-1} & -\mathbf{U}_{00}^{-1}\mathbf{U}_{01}\,\mathbf{U}_{11}^{-1} \\ \mathbf{0} & \mathbf{U}_{11}^{-1} \end{bmatrix}$$

This formula exhibits the recursive structure of the inverse of a triangular matrix and provides an efficient method for inverting large triangular matrices.
:::

:::{prf:proposition} Invertibility of Block Upper Triangular Matrices
:label: pro-block-triangular-invertible
Let $\mathbf{U} = \begin{bmatrix} \mathbf{U}_{00} & \mathbf{U}_{01} \\ \mathbf{0} & \mathbf{U}_{11} \end{bmatrix}$, where $\mathbf{U}_{00}$ and $\mathbf{U}_{11}$ are square matrices. Then $\mathbf{U}$ is invertible if and only if both $\mathbf{U}_{00}$ and $\mathbf{U}_{11}$ are invertible. The same holds for block lower triangular matrices.
:::

:::{prf:proof}
($\Leftarrow$) If $\mathbf{U}_{00}$ and $\mathbf{U}_{11}$ are both invertible, denote by $\mathbf{X}$ the matrix in {prf:ref}`ex-triangular-block-inverse`. Multiplying block by block gives $\mathbf{U}\mathbf{X} = \mathbf{X}\mathbf{U} = \mathbf{I}$, so $\mathbf{U}$ is invertible and $\mathbf{U}^{-1} = \mathbf{X}$.

($\Rightarrow$) Suppose $\mathbf{U}$ is invertible. If $\mathbf{U}_{00}$ were not invertible, there would be a nonzero vector $\mathbf{x}$ with $\mathbf{U}_{00}\mathbf{x} = \mathbf{0}$, and then

$$\mathbf{U}\begin{bmatrix}\mathbf{x}\\\mathbf{0}\end{bmatrix} = \begin{bmatrix}\mathbf{U}_{00}\mathbf{x}\\\mathbf{0}\end{bmatrix} = \mathbf{0},$$

contradicting the invertibility of $\mathbf{U}$. If $\mathbf{U}_{11}$ were not invertible, then $\mathbf{U}_{11}^\top$ would not be invertible either, so there would be a nonzero vector $\mathbf{y}$ with $\mathbf{U}_{11}^\top\mathbf{y} = \mathbf{0}$, and then

$$\mathbf{U}^\top\begin{bmatrix}\mathbf{0}\\\mathbf{y}\end{bmatrix} = \begin{bmatrix}\mathbf{U}_{00}^\top & \mathbf{0}\\ \mathbf{U}_{01}^\top & \mathbf{U}_{11}^\top\end{bmatrix}\begin{bmatrix}\mathbf{0}\\\mathbf{y}\end{bmatrix} = \begin{bmatrix}\mathbf{0}\\\mathbf{U}_{11}^\top\mathbf{y}\end{bmatrix} = \mathbf{0},$$

contradicting the invertibility of $\mathbf{U}^\top$. Hence both diagonal blocks are invertible. The transpose of a block lower triangular matrix is block upper triangular, so applying the result above to the transpose settles that case.
:::


**Existence of Inverses of Triangular Matrices**
:::{prf:theorem} Existence of Inverses of Triangular Matrices
:label: thm-triangular-matrix-invertible
For an upper (or lower) triangular square matrix, invertibility is completely determined by the diagonal entries:
- if all diagonal entries are nonzero, then the matrix is invertible, and its inverse is a triangular matrix of the same type
- if some diagonal entry is zero, then the matrix is not invertible
:::

:::{prf:proof} Existence of Inverses of Triangular Matrices
:label: prf-triangular-invertible
**The proof has two parts.**

**Part 1: invertibility when the diagonal entries are nonzero**

We prove the upper triangular case by mathematical induction (the lower triangular case is similar):

1. **Base case.** For a $1 \times 1$ upper triangular matrix $\mathbf{A} = [a_{0,0}]$, if $a_{0,0} \neq 0$, then clearly $\mathbf{A}^{-1} = [1/a_{0,0}]$, and the conclusion holds.

2. **Induction hypothesis.** Assume that every $k \times k$ ($k \geq 1$) upper triangular square matrix whose diagonal entries are all nonzero is invertible.

3. **Induction step.** Consider a $(k+1) \times (k+1)$ upper triangular square matrix in block form:

$$\mathbf{A} = \begin{bmatrix} \mathbf{A}_k & \mathbf{v} \\ \mathbf{0} & a_{k,k} \end{bmatrix}$$
where $\mathbf{A}_k$ is a $k \times k$ upper triangular submatrix, $\mathbf{v} \in \mathbb{R}^k$ is a column vector, and $a_{k,k}$ is the last diagonal entry.

4. By the block matrix inversion formula,

$$\underline{\mathbf{A}^{-1} = \begin{bmatrix} \mathbf{A}_{k}^{-1} & -\frac{1}{a_{k,k}}\mathbf{A}_{k}^{-1}\mathbf{v} \\ \mathbf{0} & \frac{1}{a_{k,k}} \end{bmatrix}}$$

5. By the induction hypothesis, if the diagonal entries of $\mathbf{A}_k$ are all nonzero, then $\mathbf{A}_k^{-1}$ exists and is upper triangular. If $a_{k,k} \neq 0$, then $\mathbf{A}^{-1}$ is also upper triangular.

**Part 2: non-invertibility when some diagonal entry is zero**

1. Let the square matrix have size $n$, and suppose that there is an index $k$ ($0 \leq k \leq n-1$) such that $a_{k,k} = 0$ and $\forall\, i \in \{0,1,\ldots,k-1\},\; a_{i,i} \neq 0$.

When $k=0$, column 0 is the zero vector, so the rank is at most $n-1$. From here on let $k>0$.

2. Considering the action of the upper triangular matrix $\mathbf{A}$ on the standard basis vectors, we obtain a **chain of nested subspaces** (a sequence of subspaces whose dimensions increase one step at a time, called a flag in the literature): the images of the first $k$ basis vectors still span a subspace of dimension exactly $k$,
$$\text{span}\{\mathbf{A}\mathbf{e}_0\} = \text{span}\{\mathbf{e}_0\} \subset \text{span}\{\mathbf{A}\mathbf{e}_0,\mathbf{A}\mathbf{e}_1\} = \text{span}\{\mathbf{e}_0,\mathbf{e}_1\} \subset \cdots \subset \text{span}\{\mathbf{A}\mathbf{e}_0,\ldots,\mathbf{A}\mathbf{e}_{k-1}\} = \text{span}\{\mathbf{e}_0,\ldots,\mathbf{e}_{k-1}\}$$
This is because the diagonal entries $a_{0,0},\ldots,a_{k-1,k-1}$ are all nonzero, so each $\mathbf{A}\mathbf{e}_i$ ($0 \leq i < k$) contributes a "new direction," one more dimension, to the image.

3. Since $a_{k,k} = 0$, the image of the $k$-th basis vector is
$$\mathbf{A}\mathbf{e}_k = \sum_{i=0}^{k-1} a_{i,k}\,\mathbf{e}_i \;\in\; \text{span}\{\mathbf{e}_0, \mathbf{e}_1, \ldots, \mathbf{e}_{k-1}\}$$

4. Hence $\mathbf{A}\mathbf{e}_k$ lies in the subspace already spanned by the first $k$ image vectors and contributes no new dimension:
$$\text{span}\{\mathbf{A}\mathbf{e}_0, \mathbf{A}\mathbf{e}_1, \ldots, \mathbf{A}\mathbf{e}_k\} = \text{span}\{\mathbf{e}_0, \mathbf{e}_1, \ldots, \mathbf{e}_{k-1}\}$$
The dimension is still $k$; it has not increased to $k+1$.

5. Let $W = \text{span}\{\mathbf{A}\mathbf{e}_0,\ldots,\mathbf{A}\mathbf{e}_{k}\}$. Then $\dim W = k$, and each of the remaining $n-k-1$ column vectors adds at most one dimension, so the dimension of the image satisfies
$$\dim\,\text{Im}(\mathbf{A}) \leq k + (n - 1 - k) = n - 1 < n$$
By the dimension theorem, $\text{rank}(\mathbf{A}) < n$; hence $\mathbf{A}$ is not surjective, and therefore $\underline{\mathbf{A} \text{ is not invertible}}$ .

**Conclusion.** The invertibility of a triangular matrix is completely determined by its diagonal entries.
:::

---

Here is a concrete example:

:::{prf:example} An Upper Triangular Matrix with a Zero on the Diagonal Is Not Invertible
:label: ex-triangular-zero-diagonal

Consider the $3 \times 3$ upper triangular matrix
$$\mathbf{A} = \begin{bmatrix} 2 & 3 & 1 \\ 0 & 0 & 5 \\ 0 & 0 & 4 \end{bmatrix}$$
Its diagonal entries are, in order, $a_{0,0}=2,\; a_{1,1}=0,\; a_{2,2}=4$, of which $a_{1,1}=0$, that is, $k=1$.

**Step 1.** Compute the images of the first $k=1$ basis vectors:
$$\mathbf{A}\mathbf{e}_0 = \begin{bmatrix}2\\0\\0\end{bmatrix} = 2\mathbf{e}_0$$
Since $a_{0,0}=2\neq 0$, $\mathbf{A}\mathbf{e}_0$ supplies a new direction, and $\dim\,\text{span}\{\mathbf{A}\mathbf{e}_0\} = 1$.

**Step 2.** Compute the image at $k=1$ (where $a_{1,1}=0$):
$$\mathbf{A}\mathbf{e}_1 = \begin{bmatrix}3\\0\\0\end{bmatrix} = 3\mathbf{e}_0 \in \text{span}\{\mathbf{e}_0\}$$
$\mathbf{A}\mathbf{e}_1$ falls into the existing subspace and does not increase the dimension, so
$$\dim\,\text{span}\{\mathbf{A}\mathbf{e}_0,\,\mathbf{A}\mathbf{e}_1\} = 1$$

**Step 3.** Compute the image at $j=2$:
$$\mathbf{A}\mathbf{e}_2 = \begin{bmatrix}1\\5\\4\end{bmatrix} \in \text{span}\{\mathbf{e}_0,\mathbf{e}_1,\mathbf{e}_2\}$$
This vector does supply a component beyond $\text{span}\{\mathbf{e}_0\}$, so the dimension of the image increases to at most $2$.

**Step 4.** Check the rank:
$$\text{rank}(\mathbf{A}) \leq 1 + 1 = 2 < 3 = n$$
(In fact one can compute that $\mathbf{A}\mathbf{e}_0$ and $\mathbf{A}\mathbf{e}_2$ are linearly independent, so $\text{rank}(\mathbf{A}) = 2$.)

**Step 5.** Since $\text{rank}(\mathbf{A}) = 2 < 3$, the matrix $\mathbf{A}$ is not invertible, which confirms the non-invertibility argument based on the chain of nested subspaces.
:::



**Group Structure**

:::{prf:property} ◇Group Properties of Triangular Matrices under Multiplication
:label: pro-triangular-group
From the analysis above we obtain the following important properties:
- The invertible upper triangular matrices (those with nonzero diagonal entries) form a group under matrix multiplication
- The unit upper triangular matrices form a group under matrix multiplication  
- The unit lower triangular matrices form a group under matrix multiplication

These group structures play important roles in matrix decompositions (such as the LU decomposition) and in numerical computation.
:::


::::{exercise} Inverse of an Upper Triangular Matrix
:label: exer-ch5-triangular-inverse

Let $\mathbf{U} = \begin{pmatrix} 1 & 2 & 3 \\ 0 & 1 & 4 \\ 0 & 0 & 2 \end{pmatrix}$ be an upper triangular matrix.

**(a)** Determine whether $\mathbf{U}$ is invertible, and explain why.

**(b)** Using the fact that "the inverse preserves the upper triangular structure," let $\mathbf{U}^{-1}$ be an upper triangular matrix with unknown entries, and solve $\mathbf{U}\mathbf{U}^{-1}=\mathbf{I}$ column by column, starting from the last column and working leftward, and from the bottom up within each column (back substitution), to find $\mathbf{U}^{-1}$.

**(c)** Verify that the diagonal entries of $\mathbf{U}^{-1}$ are exactly the reciprocals of the diagonal entries of $\mathbf{U}$.

:::{solution} exer-ch5-triangular-inverse
:class: dropdown

**(a)** The diagonal entries $1, 1, 2$ are all nonzero, so by {prf:ref}`thm-triangular-matrix-invertible`, $\mathbf{U}$ is invertible.

**(b)** Let $\mathbf{U}^{-1} = \begin{pmatrix} x_{00} & x_{01} & x_{02} \\ 0 & x_{11} & x_{12} \\ 0 & 0 & x_{22} \end{pmatrix}$. Solve $\mathbf{U}\mathbf{U}^{-1}=\mathbf{I}$ column by column:

Column 2: $2x_{22}=1 \Rightarrow x_{22}=\tfrac{1}{2}$; $x_{12}+4x_{22}=0 \Rightarrow x_{12}=-2$; $x_{02}+2x_{12}+3x_{22}=0 \Rightarrow x_{02}=\tfrac{5}{2}$.

Column 1: $x_{11}=1$; $x_{01}+2x_{11}=0 \Rightarrow x_{01}=-2$. Column 0: $x_{00}=1$. Therefore

$$
\mathbf{U}^{-1} = \begin{pmatrix} 1 & -2 & \tfrac{5}{2} \\ 0 & 1 & -2 \\ 0 & 0 & \tfrac{1}{2} \end{pmatrix}.
$$

**(c)** The diagonal of $\mathbf{U}$ is $(1,1,2)$ and the diagonal of $\mathbf{U}^{-1}$ is $(1,1,\tfrac{1}{2})$; they are reciprocals entry by entry. ✓ (This is a direct consequence of the fact that, when triangular matrices are multiplied, the diagonal entries "multiply position by position.")
:::
::::

**Summary**

The inversion of triangular matrices follows clear rules: invertibility is completely determined by the diagonal entries, and the inverse preserves the same triangular structure. The block form of the inversion formula provides an effective tool for handling large, complicated matrices, and the group structure lays an important foundation for matrix theory and numerical methods.

:::{admonition} §5.3 Summary
:class: tip
This section answers the question: how can the block structure of a large matrix be used to reduce the computational complexity of inverting it? Block diagonal matrices provide the simplest starting point—if every diagonal block is invertible, one need only invert block by block, and the off-diagonal blocks automatically remain zero, splitting the global problem into several independent small problems. For a general $2\times 2$ block matrix, the Schur complement $\mathbf{S}=\mathbf{D}-\mathbf{C}\mathbf{A}^{-1}\mathbf{B}$ is the core quantity measuring "overall invertibility": the inversion formula reduces the computation to inverting $\mathbf{A}$ and $\mathbf{S}$ separately, and the determinant factors correspondingly as $\det\mathbf{M}=\det\mathbf{A}\cdot\det\mathbf{S}$.

The invertibility of block triangular matrices is cleaner still: provided that the diagonal blocks are all square, a block triangular matrix is invertible if and only if each of its diagonal blocks is invertible; for scalar triangular matrices this is equivalent to all diagonal entries being nonzero, and the inverse preserves the same triangular structure. This conclusion is established rigorously by mathematical induction, while the argument via the chain of nested subspaces (the flag) reveals how a zero diagonal entry causes a dimension deficit in the image. The invertible upper triangular matrices of a fixed size form a group under multiplication; this algebraic structure plays a central role in numerical methods such as the LU decomposition and lays the foundation for the numerical solution of systems of linear equations in Chapter 6. §5.4 will extend the block viewpoint to higher-dimensional tensor structures and enter the domain of applications in quantum information.
:::

## 5.4 ◆Advanced Topics in Block Matrices

### 5.4.1 The Tensor Product (Kronecker Product) and Block Matrices

The tensor product is an important application of block matrices, especially in quantum mechanics and multilinear algebra. Chapter 4 ({prf:ref}`def-tensor-product-space`) already defined the tensor product space $V\otimes W$ at the level of abstract vector spaces and announced that the details of its operations would be left to this chapter. In the setting of matrix algebra, the tensor product and the Kronecker product are two names for the same concept—indeed, the space of $m\times n$ matrices is itself isomorphic to a tensor product space, $M_{m\times n}(\mathbb{R}) \cong \mathbb{R}^m \otimes \mathbb{R}^n$ (each rank-one matrix $\mathbf{u}\mathbf{v}^\top$ corresponds to the pure tensor $\mathbf{u}\otimes\mathbf{v}$), so the Kronecker product of this section is precisely the realization of the abstract tensor product in concrete matrix coordinates. Tensor product matrices have a natural block structure.

::: {prf:property} Basic Properties of the Tensor Product
:label: pro-tensor-properties
The tensor product has the following important properties:

1. **Bilinearity.** $(c\mathbf{A}) \otimes \mathbf{B} = \mathbf{A} \otimes (c\mathbf{B}) = c(\mathbf{A} \otimes \mathbf{B})$
2. **Noncommutativity.** In general, $\mathbf{A} \otimes \mathbf{B} \neq \mathbf{B} \otimes \mathbf{A}$
3. **Associativity.** $(\mathbf{A} \otimes \mathbf{B}) \otimes \mathbf{C} = \mathbf{A} \otimes (\mathbf{B} \otimes \mathbf{C})$
4. **Distributivity.** $(\mathbf{A} + \mathbf{B}) \otimes \mathbf{C} = \mathbf{A} \otimes \mathbf{C} + \mathbf{B} \otimes \mathbf{C}$
5. **Mixed-product property.** $(\mathbf{A} \otimes \mathbf{B})(\mathbf{C} \otimes \mathbf{D}) = (\mathbf{AC}) \otimes (\mathbf{BD})$ (when the matrix dimensions are compatible)
6. **Transpose property.** $(\mathbf{A} \otimes \mathbf{B})^\top = \mathbf{A}^\top \otimes \mathbf{B}^\top$
7. **Trace property.** $\text{tr}(\mathbf{A} \otimes \mathbf{B}) = \text{tr}(\mathbf{A}) \cdot \text{tr}(\mathbf{B})$
:::

These properties are very useful in practical computation—especially the mixed-product property, which allows us to break complicated tensor product operations down into simpler matrix multiplications.

::: {prf:definition} Tensor Product (Kronecker Product) of Matrices
:label: def-tensor-product
Given two matrices $\mathbf{A} \in \mathbb{R}^{m \times n}$ and $\mathbf{B} \in \mathbb{R}^{p \times q}$, their tensor product $\mathbf{A} \otimes \mathbf{B} \in \mathbb{R}^{mp \times nq}$ is defined as

$$\mathbf{A} \otimes \mathbf{B} = 
\begin{bmatrix}
a_{00}\mathbf{B} & a_{01}\mathbf{B} & \cdots & a_{0,n-1}\mathbf{B} \\
a_{10}\mathbf{B} & a_{11}\mathbf{B} & \cdots & a_{1,n-1}\mathbf{B} \\
\vdots & \vdots & \ddots & \vdots \\
a_{m-1,0}\mathbf{B} & a_{m-1,1}\mathbf{B} & \cdots & a_{m-1,n-1}\mathbf{B}
\end{bmatrix}$$

where the $a_{ij}$ are the entries of the matrix $\mathbf{A}$.
:::

As the definition shows, the tensor product naturally forms a block matrix structure in which each block is the original matrix $\mathbf{B}$ multiplied by the corresponding entry of $\mathbf{A}$.

**Connection with the 4D tensor representation.**  
Note that this block structure of the tensor product corresponds exactly to the 4D tensor representation discussed in Section 5.1.4. If the tensor product $\mathbf{A} \otimes \mathbf{B}$ is viewed as a 4D tensor, then

$$(\mathbf{A} \otimes \mathbf{B})_{ijkl} = a_{ij} \cdot b_{kl}$$

This corresponds to the notation $\mathcal{A}[i,j,k,l]$ of Section 5.1.4. Each block $a_{ij}\mathbf{B}$ can be understood as a "slice" of the 4D tensor.

::: {prf:example} Block Representation of a Tensor Product
:label: ex-tensor-matrix
Consider the two matrices

$$\mathbf{A} = \begin{bmatrix} 1 & 2 \\ 3 & 4 \end{bmatrix} 
\quad \text{and} \quad 
\mathbf{B} = \begin{bmatrix} 5 & 6 \\ 7 & 8 \end{bmatrix}$$

Their tensor product is

$$\mathbf{A} \otimes \mathbf{B} = 
\begin{bmatrix} 
1\mathbf{B} & 2\mathbf{B} \\
3\mathbf{B} & 4\mathbf{B}
\end{bmatrix} = 
\begin{bmatrix} 
5 & 6 & 10 & 12 \\
7 & 8 & 14 & 16 \\
15 & 18 & 20 & 24 \\
21 & 24 & 28 & 32
\end{bmatrix}$$

**4D tensor representation.**  
Following the indexing rules of Section 5.1.4, the 4D tensor representation of this tensor product is $\mathcal{T} \in \mathbb{R}^{2 \times 2 \times 2 \times 2}$, where
- $\mathcal{T}[0,0,:,:] = 1 \times \mathbf{B} = \begin{bmatrix} 5 & 6 \\ 7 & 8 \end{bmatrix}$
- $\mathcal{T}[0,1,:,:] = 2 \times \mathbf{B} = \begin{bmatrix} 10 & 12 \\ 14 & 16 \end{bmatrix}$
- $\mathcal{T}[1,0,:,:] = 3 \times \mathbf{B} = \begin{bmatrix} 15 & 18 \\ 21 & 24 \end{bmatrix}$
- $\mathcal{T}[1,1,:,:] = 4 \times \mathbf{B} = \begin{bmatrix} 20 & 24 \\ 28 & 32 \end{bmatrix}$
:::

#### Tensor Products of Vectors

When we work with vectors, the tensor product can likewise be represented by a block structure. If $\mathbf{v} \in \mathbb{R}^m$ and $\mathbf{w} \in \mathbb{R}^n$, then $\mathbf{v} \otimes \mathbf{w} \in \mathbb{R}^{mn}$:

$$\mathbf{v} \otimes \mathbf{w} = 
\begin{bmatrix}
v_0\mathbf{w} \\
v_1\mathbf{w} \\
\vdots \\
v_{m-1}\mathbf{w}
\end{bmatrix}$$

This shows that the tensor product of vectors can be viewed as a special kind of block matrix. In the framework of 4D tensors, it can be regarded as a degenerate case in which some axes have size 1.

::: {prf:example} Tensor Product of Vectors
:label: ex-tensor-vector
Consider the vectors $\mathbf{v} = (1, 2)^\top$ and $\mathbf{w} = (3, 4)^\top$; their tensor product is

$$\mathbf{v} \otimes \mathbf{w} = 
\begin{bmatrix}
1\mathbf{w} \\
2\mathbf{w}
\end{bmatrix} = 
\begin{bmatrix}
3 \\
4 \\
6 \\
8
\end{bmatrix}$$

**Index correspondence.**  
If this result is rearranged into a $2 \times 2$ matrix in NumPy's C order, we obtain
$$\begin{bmatrix} 3 & 4 \\ 6 & 8 \end{bmatrix}$$
This corresponds to the 4D tensor representation $\mathcal{T}[i,0,j,0] = v_i \cdot w_j$.
:::



#### Advantages of the Block Matrix Representation

From a computational point of view, the block representation of the tensor product has several important advantages:

1. **Clear structure.** The block representation shows directly how the tensor product is constructed and corresponds naturally to the 4D tensor indexing of Section 5.1.4
2. **Computational convenience.** The block structure can be used to design more efficient parallel algorithms, especially in combination with batched operations on 4D tensors
3. **Memory management.** For large sparse matrices, the block representation can save storage space
4. **Numerical stability.** Block operations help control the accumulation of numerical errors

```{note}
**A unified mathematical framework.**  
Combined with the 4D tensor representation of Section 5.1.4, we can build a unified mathematical framework for handling block matrices, tensor products, and high-dimensional data structures. This unity is important in modern machine learning and quantum computing.

**Optimizing computation.** In practice we usually do not actually construct the full tensor product matrix; instead we exploit its block structure and its 4D tensor representation to compute efficiently, which is especially important when handling high-dimensional data.
```

#### Applications in Quantum Mechanics

In quantum mechanics, the tensor product is the basic tool for describing composite quantum systems. When two quantum systems are combined, the total state space is the tensor product of the state spaces of the subsystems. Combined with the 4D tensor representation of Section 5.1.4, we can handle these high-dimensional structures more efficiently, especially when computing the various operations involved in quantum entanglement and quantum information processing.

---


::::{exercise} Numerical Verification of the Mixed-Product Property
:label: exer-ch5-kron-mixed

Let $\mathbf{A} = \begin{pmatrix} 1 & 2 \\ 0 & 1 \end{pmatrix}$, $\mathbf{B} = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$, $\mathbf{C} = \begin{pmatrix} 1 & 0 \\ 1 & 1 \end{pmatrix}$, and $\mathbf{D} = \begin{pmatrix} 2 & 0 \\ 0 & 3 \end{pmatrix}$.

**(a)** Compute $\mathbf{AC}$ and $\mathbf{BD}$ by hand, and write down the upper-left $2\times 2$ block of $(\mathbf{AC})\otimes(\mathbf{BD})$.

**(b)** Verify the mixed-product property $(\mathbf{A}\otimes\mathbf{B})(\mathbf{C}\otimes\mathbf{D}) = (\mathbf{AC})\otimes(\mathbf{BD})$ with `np.kron`, and write down the key code.

**(c)** Explain why the mixed-product property implies that, when the tensor product $\mathbf{A}\otimes\mathbf{B}$ of two $n\times n$ matrices acts on a vector, one can avoid explicitly constructing the large $n^2\times n^2$ matrix.

:::{solution} exer-ch5-kron-mixed
:class: dropdown

**(a)** $\mathbf{AC} = \begin{pmatrix} 3 & 2 \\ 1 & 1 \end{pmatrix}$ and $\mathbf{BD} = \begin{pmatrix} 0 & 3 \\ 2 & 0 \end{pmatrix}$. The upper-left $2\times 2$ block of $(\mathbf{AC})\otimes(\mathbf{BD})$ is $(\mathbf{AC})_{00}\cdot\mathbf{BD} = 3\begin{pmatrix} 0 & 3 \\ 2 & 0 \end{pmatrix} = \begin{pmatrix} 0 & 9 \\ 6 & 0 \end{pmatrix}$.

**(b)**

```python
import numpy as np
A = np.array([[1, 2], [0, 1]]); B = np.array([[0, 1], [1, 0]])
C = np.array([[1, 0], [1, 1]]); D = np.array([[2, 0], [0, 3]])
lhs = np.kron(A, B) @ np.kron(C, D)
rhs = np.kron(A @ C, B @ D)
print(np.allclose(lhs, rhs))   # True
```

**(c)** Fix column-by-column stacking explicitly: $\mathrm{vec}_F(\mathbf{X})=\mathbf{X}.\mathrm{reshape}(-1,\mathrm{order}='F')$. For $n\times n$ matrices,
$$ (\mathbf{A}\otimes\mathbf{B})\mathrm{vec}_F(\mathbf{X})=\mathrm{vec}_F(\mathbf{B}\mathbf{X}\mathbf{A}^{\top}). $$
In code, first form `X = v.reshape(n, n, order='F')`, then `(B @ X @ A.T).reshape(-1, order='F')`. The cost drops from $O(n^4)$ to $O(n^3)$, and there is no need to store an $n^2\times n^2$ matrix. Note that this F-order differs from the C-order rearrangement used for display earlier.
:::
::::


### 5.4.2 Trace and Partial Trace of Block Matrices

The trace and the partial trace are important when dealing with matrices that have a tensor product structure, especially in quantum information and the analysis of many-body systems. Building on the 4D tensor representation of Section 5.1.4, this section systematically introduces how these operations are computed.

#### 5.4.2.1 The Trace of a Block Matrix

::: {prf:definition} Trace of a Block Matrix
:label: def-block-trace-new
Given a square block matrix

$$\mathbf{A} = 
\begin{bmatrix}
\mathbf{A}_{00} & \mathbf{A}_{01} & \cdots & \mathbf{A}_{0,n-1} \\
\mathbf{A}_{10} & \mathbf{A}_{11} & \cdots & \mathbf{A}_{1,n-1} \\
\vdots & \vdots & \ddots & \vdots \\
\mathbf{A}_{n-1,0} & \mathbf{A}_{n-1,1} & \cdots & \mathbf{A}_{n-1,n-1}
\end{bmatrix}$$

in which every diagonal block $\mathbf{A}_{ii}$ is square, the trace of the matrix is

$$\text{tr}(\mathbf{A}) = \sum_{i=0}^{n-1} \text{tr}(\mathbf{A}_{ii})$$

**4D tensor representation.** In the notation of Section 5.1.4, if $\mathbf{A}$ corresponds to the 4D tensor $\mathcal{A}[i,j,k,l]$, then
$$\text{tr}(\mathbf{A}) = \sum_{i=0}^{n-1} \sum_{k=0}^{p-1} \mathcal{A}[i,i,k,k]$$
where $p$ is the size of each diagonal block.
:::

For tensor product matrices, the trace has a particularly simple property:

::: {prf:property} Trace of a Tensor Product Matrix
:label: pro-trace-tensor-new
If $\mathbf{A}$ and $\mathbf{B}$ are $m \times m$ and $n \times n$ matrices, respectively, then

$$\text{tr}(\mathbf{A} \otimes \mathbf{B}) = \text{tr}(\mathbf{A}) \cdot \text{tr}(\mathbf{B})$$

**Proof.** The diagonal entries of the tensor product are $a_{ii} \cdot b_{jj}$, so
$$\text{tr}(\mathbf{A} \otimes \mathbf{B}) = \sum_{i,j} a_{ii} b_{jj} = \left(\sum_i a_{ii}\right)\left(\sum_j b_{jj}\right) = \text{tr}(\mathbf{A}) \cdot \text{tr}(\mathbf{B})$$
:::

#### 5.4.2.2 Definition and Computation of the Partial Trace

The partial trace is the natural generalization of the trace to tensor product structures, used to take a trace "partially." Using the 4D tensor representation of Section 5.1.4, we can give a precise definition of the partial trace.

::: {prf:definition} 4D Tensor Definition of the Partial Trace
:label: def-partial-trace-4d-final
Consider a matrix $\mathbf{M}$ representing a composite system, with 4D tensor representation $\mathcal{M}[i,j,k,l]$. By the indexing rules of Section 5.1.4:
- **System A** is described by the indices $(i,j)$ (the position of the block)
- **System B** is described by the indices $(k,l)$ (the position within the block)

**Partial trace over system A** (keeping system B):
$$[\text{Tr}_A(\mathbf{M})]_{kl} = \sum_{i=0}^{m-1} \mathcal{M}[i,i,k,l]$$

**Partial trace over system B** (keeping system A):
$$[\text{Tr}_B(\mathbf{M})]_{ij} = \sum_{k=0}^{n-1} \mathcal{M}[i,j,k,k]$$

where $m$ and $n$ are the dimensions of the two subsystems.
:::

```{note}
**The essence of the dimension reduction.**
- **Input**: the 4D tensor $\mathcal{M}[i,j,k,l]$ (it describes a general operator on the composite space, and describes a quantum state only if it is a density matrix)
- **Partial trace over system A**: yields a 2D matrix depending only on the indices $(k,l)$; when the input is a density matrix, it describes the reduced state of system B
- **Partial trace over system B**: yields a 2D matrix depending only on the indices $(i,j)$; when the input is a density matrix, it describes the reduced state of system A

The core of the partial trace operation is **fixing and summing diagonal entries**:
- Partial trace over A: set axis 0 equal to axis 1 ($i=j$) and sum
- Partial trace over B: set axis 2 equal to axis 3 ($k=l$) and sum
```

#### Computing the Partial Trace from the Block Matrix Viewpoint

From a practical computational standpoint, block matrices provide a more intuitive method:

::: {prf:property} Block Formula for the Partial Trace
:label: pro-partial-trace-block-final
Write the matrix $\mathbf{M}$ as a block matrix indexed by system A:

$$\mathbf{M} = 
\begin{bmatrix}
\mathbf{M}_{00}^{(B)} & \mathbf{M}_{01}^{(B)} & \cdots & \mathbf{M}_{0,m-1}^{(B)} \\
\mathbf{M}_{10}^{(B)} & \mathbf{M}_{11}^{(B)} & \cdots & \mathbf{M}_{1,m-1}^{(B)} \\
\vdots & \vdots & \ddots & \vdots \\
\mathbf{M}_{m-1,0}^{(B)} & \mathbf{M}_{m-1,1}^{(B)} & \cdots & \mathbf{M}_{m-1,m-1}^{(B)}
\end{bmatrix}$$

where each $\mathbf{M}_{ij}^{(B)}$ is an $n \times n$ matrix describing system B. Then:

**Partial trace over system A** (keeping system B):

$$\text{Tr}_A(\mathbf{M}) = \sum_{i=0}^{m-1} \mathbf{M}_{ii}^{(B)}$$
*The result is an $n \times n$ matrix; when the input is a density matrix, it describes the reduced state of system B*

**Partial trace over system B** (keeping system A):

$$\text{Tr}_B(\mathbf{M}) = 
\begin{bmatrix}
\text{tr}(\mathbf{M}_{00}^{(B)}) & \text{tr}(\mathbf{M}_{01}^{(B)}) & \cdots & \text{tr}(\mathbf{M}_{0,m-1}^{(B)}) \\
\text{tr}(\mathbf{M}_{10}^{(B)}) & \text{tr}(\mathbf{M}_{11}^{(B)}) & \cdots & \text{tr}(\mathbf{M}_{1,m-1}^{(B)}) \\
\vdots & \vdots & \ddots & \vdots \\
\text{tr}(\mathbf{M}_{m-1,0}^{(B)}) & \text{tr}(\mathbf{M}_{m-1,1}^{(B)}) & \cdots & \text{tr}(\mathbf{M}_{m-1,m-1}^{(B)})
\end{bmatrix}$$
*The result is an $m \times m$ matrix; when the input is a density matrix, it describes the reduced state of system A*
:::

#### Correspondence between the 4D Tensor and Block Computations

::: {prf:property} Consistency of the Tensor and Block Representations
:label: pro-tensor-block-consistency
By the indexing rules of Section 5.1.4, the entries $M_{rs}$ of the original matrix correspond to the entries of the 4D tensor as follows:
$$\mathcal{M}[i,j,k,l] = M_{in+k, jn+l}$$

where $n$ is the dimension of the second subsystem, which is also the stride of the combined index.

**Consistency of the partial trace over system A.**
$$[\text{Tr}_A(\mathbf{M})]_{kl} = \sum_{i=0}^{m-1} \mathcal{M}[i,i,k,l] = \sum_{i=0}^{m-1} M_{in+k, in+l}$$

This corresponds exactly to the sum of the $(k,l)$ entries of the diagonal blocks in the block computation:
$$[\sum_{i=0}^{m-1} \mathbf{M}_{ii}^{(B)}]_{kl} = \sum_{i=0}^{m-1} [\mathbf{M}_{ii}^{(B)}]_{kl} = \sum_{i=0}^{m-1} M_{in+k, in+l}$$
:::

#### A Complete Worked Example

::: {prf:example} Detailed Computation of the Partial Traces of a 4×4 Matrix
:label: ex-partial-trace-detailed
Consider the 4×4 matrix (two 2-dimensional subsystems)

$$\mathbf{M} = \begin{bmatrix}
1 & 2 & 3 & 4 \\
5 & 6 & 7 & 8 \\
9 & 10 & 11 & 12 \\
13 & 14 & 15 & 16
\end{bmatrix}$$

**Step 1. 4D tensor representation**

By the indexing rule $\mathcal{M}[i,j,k,l] = M_{2i+k, 2j+l}$:
- $\mathcal{M}[0,0,0,0] = M_{00} = 1$
- $\mathcal{M}[0,0,0,1] = M_{01} = 2$
- $\mathcal{M}[1,1,0,0] = M_{22} = 11$
- $\mathcal{M}[1,1,1,1] = M_{33} = 16$
- ...

**Step 2. Partial trace over system A (4D tensor method)**

$$[\text{Tr}_A(\mathbf{M})]_{kl} = \sum_{i=0}^{1} \mathcal{M}[i,i,k,l] = \mathcal{M}[0,0,k,l] + \mathcal{M}[1,1,k,l]$$

Computing entry by entry:
- $[\text{Tr}_A(\mathbf{M})]_{00} = \mathcal{M}[0,0,0,0] + \mathcal{M}[1,1,0,0] = M_{00} + M_{22} = 1 + 11 = 12$
- $[\text{Tr}_A(\mathbf{M})]_{01} = \mathcal{M}[0,0,0,1] + \mathcal{M}[1,1,0,1] = M_{01} + M_{23} = 2 + 12 = 14$
- $[\text{Tr}_A(\mathbf{M})]_{10} = \mathcal{M}[0,0,1,0] + \mathcal{M}[1,1,1,0] = M_{10} + M_{32} = 5 + 15 = 20$
- $[\text{Tr}_A(\mathbf{M})]_{11} = \mathcal{M}[0,0,1,1] + \mathcal{M}[1,1,1,1] = M_{11} + M_{33} = 6 + 16 = 22$

Therefore $$\text{Tr}_A(\mathbf{M}) = \begin{bmatrix} 12 & 14 \\ 20 & 22 \end{bmatrix}$$

**Step 3. Check by block computation**

Partition the matrix into 2×2 blocks:

$$\mathbf{M} = \begin{bmatrix}
\mathbf{M}_{00}^{(B)} & \mathbf{M}_{01}^{(B)} \\[3mm]
\mathbf{M}_{10}^{(B)} & \mathbf{M}_{11}^{(B)}
\end{bmatrix} = \begin{bmatrix}
\begin{bmatrix} 1 & 2 \\ 5 & 6 \end{bmatrix} & \begin{bmatrix} 3 & 4 \\ 7 & 8 \end{bmatrix} \\[3mm]
\begin{bmatrix} 9 & 10 \\ 13 & 14 \end{bmatrix} & \begin{bmatrix} 11 & 12 \\ 15 & 16 \end{bmatrix}
\end{bmatrix}$$

The partial trace over system A:

$$\text{Tr}_A(\mathbf{M}) = \mathbf{M}_{00}^{(B)} + \mathbf{M}_{11}^{(B)} = \begin{bmatrix} 1 & 2 \\ 5 & 6 \end{bmatrix} + \begin{bmatrix} 11 & 12 \\ 15 & 16 \end{bmatrix} = \begin{bmatrix} 12 & 14 \\ 20 & 22 \end{bmatrix}$$

**The results agree** ✓

**Step 4. Partial trace over system B (4D tensor method)**

$$[\text{Tr}_B(\mathbf{M})]_{ij} = \sum_{k=0}^{1} \mathcal{M}[i,j,k,k] = \mathcal{M}[i,j,0,0] + \mathcal{M}[i,j,1,1]$$

Computing entry by entry:
- $[\text{Tr}_B(\mathbf{M})]_{00} = \mathcal{M}[0,0,0,0] + \mathcal{M}[0,0,1,1] = M_{00} + M_{11} = 1 + 6 = 7$
- $[\text{Tr}_B(\mathbf{M})]_{01} = \mathcal{M}[0,1,0,0] + \mathcal{M}[0,1,1,1] = M_{02} + M_{13} = 3 + 8 = 11$
- $[\text{Tr}_B(\mathbf{M})]_{10} = \mathcal{M}[1,0,0,0] + \mathcal{M}[1,0,1,1] = M_{20} + M_{31} = 9 + 14 = 23$
- $[\text{Tr}_B(\mathbf{M})]_{11} = \mathcal{M}[1,1,0,0] + \mathcal{M}[1,1,1,1] = M_{22} + M_{33} = 11 + 16 = 27$

Therefore $$\text{Tr}_B(\mathbf{M}) = \begin{bmatrix} 7 & 11 \\ 23 & 27 \end{bmatrix}$$

**Step 5. Check by block computation**

The partial trace over system B requires the trace of each block:
- $\text{tr}(\mathbf{M}_{00}^{(B)}) = \text{tr}\begin{bmatrix} 1 & 2 \\ 5 & 6 \end{bmatrix} = 1 + 6 = 7$
- $\text{tr}(\mathbf{M}_{01}^{(B)}) = \text{tr}\begin{bmatrix} 3 & 4 \\ 7 & 8 \end{bmatrix} = 3 + 8 = 11$
- $\text{tr}(\mathbf{M}_{10}^{(B)}) = \text{tr}\begin{bmatrix} 9 & 10 \\ 13 & 14 \end{bmatrix} = 9 + 14 = 23$
- $\text{tr}(\mathbf{M}_{11}^{(B)}) = \text{tr}\begin{bmatrix} 11 & 12 \\ 15 & 16 \end{bmatrix} = 11 + 16 = 27$

Therefore
$$\text{Tr}_B(\mathbf{M}) = \begin{bmatrix} 7 & 11 \\ 23 & 27 \end{bmatrix}$$

**The results agree** ✓
:::


::::{exercise} Partial Traces of a $4\times 4$ Matrix
:label: exer-ch5-partial-trace

Regard $\mathbf{M} = \begin{pmatrix} 1 & 2 & 3 & 4 \\ 5 & 6 & 7 & 8 \\ 9 & 10 & 11 & 12 \\ 13 & 14 & 15 & 16 \end{pmatrix}$ as the matrix of a composite system, partitioned into $2\times 2$ blocks (each block $2\times 2$).

**(a)** Compute $\mathrm{Tr}_A(\mathbf{M})$ (the sum of the diagonal blocks).

**(b)** Compute $\mathrm{Tr}_B(\mathbf{M})$ (the trace of each block).

**(c)** Verify the consistency relation $\mathrm{tr}(\mathrm{Tr}_A(\mathbf{M})) = \mathrm{tr}(\mathrm{Tr}_B(\mathbf{M})) = \mathrm{tr}(\mathbf{M})$.

:::{solution} exer-ch5-partial-trace
:class: dropdown

The blocks are $\mathbf{M}_{00}=\begin{pmatrix}1&2\\5&6\end{pmatrix}$, $\mathbf{M}_{01}=\begin{pmatrix}3&4\\7&8\end{pmatrix}$, $\mathbf{M}_{10}=\begin{pmatrix}9&10\\13&14\end{pmatrix}$, and $\mathbf{M}_{11}=\begin{pmatrix}11&12\\15&16\end{pmatrix}$.

**(a)** $\mathrm{Tr}_A(\mathbf{M}) = \mathbf{M}_{00}+\mathbf{M}_{11} = \begin{pmatrix} 12 & 14 \\ 20 & 22 \end{pmatrix}$.

**(b)** The $(i,j)$ entry of $\mathrm{Tr}_B(\mathbf{M})$ is $\mathrm{tr}(\mathbf{M}_{ij})$: $\mathrm{tr}(\mathbf{M}_{00})=7$, $\mathrm{tr}(\mathbf{M}_{01})=11$, $\mathrm{tr}(\mathbf{M}_{10})=23$, and $\mathrm{tr}(\mathbf{M}_{11})=27$, so $\mathrm{Tr}_B(\mathbf{M}) = \begin{pmatrix} 7 & 11 \\ 23 & 27 \end{pmatrix}$.

**(c)** $\mathrm{tr}(\mathrm{Tr}_A(\mathbf{M})) = 12+22 = 34$; $\mathrm{tr}(\mathrm{Tr}_B(\mathbf{M})) = 7+27 = 34$; $\mathrm{tr}(\mathbf{M}) = 1+6+11+16 = 34$. All three agree ✓—no matter which subsystem is "compressed" first, the total trace is unchanged; this is the trace-preservation identity for general operators. The $\mathbf{M}$ in this example is not Hermitian and has trace 34, so it is not a quantum state. Only density matrices—Hermitian, positive semidefinite, and of trace 1—admit a probabilistic interpretation. For example, both partial traces of $\boldsymbol{\rho}=\mathbf{I}_4/4$ equal $\mathbf{I}_2/2$, and both are valid reduced density matrices.

(This can be checked in one line with the 4D tensor: `np.einsum('iikl->kl', M4d)` gives $\mathrm{Tr}_A$, and `np.einsum('ijkk->ij', M4d)` gives $\mathrm{Tr}_B$.)
:::
::::


:::{admonition} §5.4 Summary
:class: tip
This section answers the questions: how can the language of block matrices describe a composite system, and how can information about a single subsystem be "extracted" from a composite system? The Kronecker product $\mathbf{A}\otimes\mathbf{B}$ is block thinking taken to its limit: using the entries of $\mathbf{A}$ as scalar coefficients, it scales copies of $\mathbf{B}$ block by block and assembles them into a higher-dimensional matrix. The mixed-product property $(\mathbf{A}\otimes\mathbf{B})(\mathbf{C}\otimes\mathbf{D})=(\mathbf{AC})\otimes(\mathbf{BD})$ shows that when the evolution factors into local operators $\mathbf{U}_A\otimes\mathbf{U}_B$, the local actions can be computed separately; in the presence of interactions, one generally has to use general operators on the whole tensor product space.

The partial trace is a reduction operation that eliminates the degrees of freedom of one subsystem and is in general not invertible; only for a product state in which the factor being traced out has trace 1 can the other factor be recovered. $\mathrm{Tr}_A(\mathbf{M})=\sum_i\mathbf{M}_{ii}^{(B)}$ sums the diagonal blocks, compressing the statistical information of subsystem $B$ out of the high-dimensional description of the composite system; it is indispensable in measuring quantum entanglement and in designing quantum communication protocols. Combined with the 4D tensor viewpoint of §5.1, the partial trace is in essence nothing more than summing the diagonal entries along specified axes; the two representations agree completely, confirming the inner conceptual consistency of this chapter, from its basic definitions to its advanced applications.
:::

## 5.5 Chapter Summary

### Review of the Theoretical Thread

This chapter started from a simple question: when a matrix is too large to handle directly, can its internal structure become the way forward for computation? The four sections unfolded in turn along a single logical thread, each answering the question left open by the one before.

§5.1 laid the foundation of the language. Although the rules for addition, scalar multiplication, and transposition of block matrices are entirely analogous to those for ordinary matrices, the extra condition of "compatibility of block sizes" runs through all of them and cannot be ignored. The 4D tensor viewpoint $\mathcal{A}[i,j,k,l]$ further gives the block indices and the within-block entry indices a single unified address, which both corresponds directly to batched computation in NumPy and lays down the conceptual interface for the Kronecker product later on.

§5.2 revealed the true power of the language of blocks. Matrix multiplication has four complementary geometric intuitions—inner products, column combinations, row combinations, and outer-product expansion—and all four converge in the block framework to one and the same formula: $(\mathbf{AB})_{ij}=\sum_k\mathbf{A}_{ik}\mathbf{B}_{kj}$. This formula, which formally "changes nothing," nevertheless enabled Strassen, with a $2\times 2$ block technique, to lower the computational complexity of matrix multiplication from $n^3$ to $n^{2.807}$, overturning an intuition a century and a half old.

§5.3 built on multiplication to discuss strategies for inverting in blocks. From the block-by-block inversion of block diagonal matrices, to the central role played by the Schur complement $\mathbf{S}=\mathbf{D}-\mathbf{C}\mathbf{A}^{-1}\mathbf{B}$ in a general $2\times 2$ block structure, to the rigorous characterization of triangular matrices, for which nonzero diagonal entries are the sole condition for invertibility, all three make the same point: the structure of inversion depends on the structure of multiplication. The invertible upper triangular matrices of a fixed size form a group under multiplication, and this algebraic property directly drives the correctness argument for the LU decomposition in numerical linear algebra.

§5.4 extended block thinking to higher dimensions. The Kronecker product scales the whole of $\mathbf{B}$ by each entry of $\mathbf{A}$, used as a scalar coefficient, constructing the complete state space of a composite system; when the evolution factors as $\mathbf{U}_A\otimes\mathbf{U}_B$, the mixed-product property allows the local actions to be computed separately, whereas general interactions admit no such factorization. The partial trace, in turn, performs a reduction, projecting the statistical information of a single subsystem out of the high-dimensional description of the composite system; in the language of 4D tensors this operation is nothing more than "summing the diagonal entries along specified axes," echoing the viewpoint of §5.1 and closing the conceptual loop of the whole chapter.

### Connections to Other Chapters

The logical starting point of this chapter is Chapter 4's abstract discussion of the structure of linear spaces. Abstract linear spaces established the equivalence among vectors, linear mappings, and matrices; block matrices are a "coarse-graining" of this equivalence—binding several dimensions together into one algebraic unit, so that a matrix of matrices becomes a legitimate mathematical object. The dimension-matching condition of block multiplication is precisely the concrete expression, in the language of blocks, of the compatibility of spaces that allows linear mappings to be composed.

Chapter 6 will take up the systematic study of systems of linear equations, and §5.3 has already laid the groundwork for its most important computational tools. The Schur complement appears in statistics as the computational core of partial correlation coefficients and in control theory as the solution structure of the Riccati equation, and above all it is the algebraic essence of the block form of Gaussian elimination. The invertibility theorem and the group structure of triangular matrices correspond directly to the correctness guarantees of the forward substitution and back substitution algorithms. This connection will reappear in a clearer form when Chapter 6 discusses the LU decomposition.

### The Role of This Chapter in the Book

The core problem that block matrices solve is how to turn a problem that is "too large" into one that "can be handled." This problem is more universal than linear algebra itself: during World War II, human computers processed the strength matrices of aircraft wings by hand in batches; today, GPU parallel computing carries out the gradient updates of deep learning in batches; and quantum computers store the density matrices of composite systems in a tensor product structure—the same way of thinking reappears in different guises across different eras.

The shock of Strassen's algorithm lies not only in the few multiplications it saved but even more in what it tells us: even for a "trivial" operation, once its internal structure is seen, a cleverer path may exist. This insight directly gave rise to the research area of "lower bounds on the complexity of matrix multiplication," which remains open to this day. If you continue reading with the block viewpoint built in this chapter, you will find that the determinant expansions, eigendecompositions, and singular value decompositions of later chapters are all, in some sense, extensions of one and the same strategy: "see the structure, exploit the structure."

### Concept Map
```{mermaid}
%%{init: {
  "theme": "base",
  "themeVariables": {
    "background":         "#FFFFFF",
    "primaryColor":       "#FFFFFF",
    "primaryTextColor":   "#57068C",
    "primaryBorderColor": "#57068C",
    "lineColor":          "#7F8C8D",
    "secondaryColor":     "#F1F2F6",
    "tertiaryColor":      "#FFFFFF",
    "clusterBkg":         "#F9F9FB",
    "clusterBorder":      "#DCDDE1",
    "fontSize":           "14px"
  }
}}%%
graph TD
    n_ch4["Abstract Linear Spaces<br/>Chapter 4"]

    subgraph S1 ["<span style='color:#57068C;font-weight:bold'>§5.1 Definition and Basic Operations</span>"]
        n_def["Definition of Block Matrices"]
        n_ops["Addition · Scalar Multiplication · Transpose"]
        n_tensor4d["4D Tensor View<br/>A[i,j,k,l]"]
    end

    subgraph S2 ["<span style='color:#57068C;font-weight:bold'>§5.2 Block Multiplication</span>"]
        n_views["Inner Product · Outer Product<br/>Linear Combinations"]
        n_blockmul["Block Multiplication Formula<br/>(AB)ij = Σ Aik Bkj"]
    end

    subgraph S3 ["<span style='color:#57068C;font-weight:bold'>§5.3 Inverse Matrices</span>"]
        n_diaginv["Block Diagonal Inverse"]
        n_schur["Schur Complement"]
        n_tri["Triangular Matrices<br/>Invertibility and Group Structure"]
    end

    subgraph S4 ["<span style='color:#57068C;font-weight:bold'>§5.4 Advanced Topics</span>"]
        n_kronecker["Kronecker Product<br/>Mixed-Product Property"]
        n_ptrace["Partial Trace<br/>Subsystem Reduction"]
    end

    n_ch6["Systems of Linear Equations<br/>Chapter 6"]

    n_ch4 -.->|builds on| n_def
    n_def --> n_ops
    n_def --> n_tensor4d
    n_ops --> n_views
    n_views --> n_blockmul
    n_tensor4d --> n_blockmul
    n_blockmul --> n_diaginv
    n_blockmul --> n_schur
    n_schur --> n_tri
    n_tensor4d --> n_kronecker
    n_kronecker --> n_ptrace
    n_schur -.->|prepares for| n_ch6
    n_tri -.->|prepares for| n_ch6

    classDef main      fill:#F4EEF9,stroke:#57068C,stroke-width:2px,color:#57068C
    classDef structure fill:#E6F0F3,stroke:#006385,stroke-width:2px,color:#006385
    classDef accent    fill:#EAF9F8,stroke:#2AD2C9,stroke-width:2px,color:#087D77,font-weight:bold
    classDef neutral   fill:#FFFFFF,stroke:#DCDDE1,color:#2F3640
    classDef future    fill:#FFFFFF,stroke:#AB82C5,stroke-dasharray:5 5,color:#7F8C8D

    class n_def main
    class n_blockmul main
    class n_tensor4d accent
    class n_schur accent
    class n_ptrace accent
    class n_ops structure
    class n_views structure
    class n_kronecker structure
    class n_diaginv neutral
    class n_tri neutral
    class n_ch4 future
    class n_ch6 future
```